# MNIST digit `1` — frozen FIT-ranked UCB policy + pixel-space direction extraction

**This notebook integrates our established November 2 scale-repair procedure with Task 2. It does *not* invent another threshold-ranking or calibration algorithm.**

**Main goal:** Load the *previously fitted and calibrated* MNIST policy; apply it unchanged to **new, disjoint digit-1 test images**; save matrix-free score-filtered directions at **t=5**, **t=100**, and the **calibrated selected timestep** (if accepted). Independently assess stability and handwriting-reference retention. No surrogate classifier or evaluation classifier is used during selection/generation.

**Separate scientific question:** Can we certify *true* tangent-projector error on real MNIST? **Not from MNIST class labels alone**: the true manifold tangent projector is unobserved. The original MNIST UCB calibrates bounded *proxy risk*, not true-error. This notebook audits that limitation explicitly. A later experiment needs independent validated tangent ground truth, or a justified additional assumption; synthetic true-error conformal coverage is not transferable by assertion.

**What you need:** A Colab T4 GPU and your *existing* Google Drive folder `MyDrive/MNIST_scale_repair` with the previous 20-FIT / 100-CAL outputs. If missing, an optional guarded rebuild uses the **exact existing original repair script** and checkpoints per image. No file upload prompts.

## A. Dependencies and preserved source implementation
Select **Runtime → Change runtime type → T4 GPU**. The next four source cells are embedded unchanged from the earlier notebooks; no uploads required.

In [ ]:
!pip -q install "diffusers==0.38.0" "huggingface_hub>=0.28" "safetensors>=0.4" "accelerate>=1.0" "torchvision"

In [ ]:
%%writefile /content/MNIST_Digit1_Final_Geometry.py
#!/usr/bin/env python3
"""Reproducible single-image MNIST digit-1 feasibility experiment (NOT official LHSD).

Pretrained class-conditional DDPM: harveymannering/mnist-ddpm.
Numerical operator: Q_t = sqrt(1-alpha_bar_t) * sym(J_{x_t} eps_theta),
where x_t=sqrt(alpha_bar_t)*x_clean and conditioning class is 1.
This equals -sigma_VE**2 sym(J_{x_clean} s_theta) with
sigma_VE=sqrt((1-alpha_bar_t)/alpha_bar_t).

Jacobian-vector products use 2 forward passes (central difference); adjoint J^T v
uses a FIRST derivative only, avoiding fused-attention double-backward errors.

Selector constants were frozen in the earlier constructed-score benchmark.
This is NOT the official published LHSD implementation, and MNIST LID has no
known ground-truth dimension. Independent translations/nearest-neighbor tests
provide heuristic geometry checks, not certified tangent ground truth.
"""
import argparse
import hashlib
import json
import math
import os
import time
from pathlib import Path
from types import SimpleNamespace
import numpy as np
import pandas as pd
import torch
from torch.nn import functional as F

MODEL_ID = 'harveymannering/mnist-ddpm'
# Frozen constants. No MNIST-label or validation-set-based tuning.
SELECT = dict(transition=0.12, defect=0.24, cross=0.28, asym=0.06, range=0.12,
              spectral_cutoff=0.10, hill_power=4, min_run=2)
SIGMAS_T = [5, 15, 35, 65, 100]


def config_for(mode):
    if mode == 'pilot':
        return dict(timesteps=[5, 35, 100], lanczos_steps=10, trace_probes=2,
                    geom_probes=2, filter_steps=10, fd_eps=0.04, ref_neighbors=60)
    if mode == 'expanded':
        return dict(timesteps=SIGMAS_T, lanczos_steps=20, trace_probes=4,
                    geom_probes=4, filter_steps=20, fd_eps=0.04, ref_neighbors=80)
    raise ValueError(mode)


def alpha_cumprod(t):
    # Author's model card: linear DDPM betas 1e-4..0.02 over 1000 steps
    betas = torch.linspace(0.0001, 0.02, 1000, dtype=torch.float32)
    return float(torch.cumprod(1.0-betas, dim=0)[int(t)])


def model_load(device):
    try:
        from diffusers import UNet2DModel
    except Exception as ex:
        raise RuntimeError('Install diffusers and huggingface_hub first.') from ex
    net = UNet2DModel.from_pretrained(MODEL_ID, use_safetensors=True).to(device).eval()
    for p in net.parameters(): p.requires_grad_(False)
    assert net.config.num_class_embeds == 11, f'Expected conditional checkpoint (11 classes), got {net.config.num_class_embeds}'
    assert net.config.in_channels == 1 and net.config.sample_size == 28, 'Expected 1x28x28 images'
    return net


def image_load(idx):
    from torchvision.datasets import MNIST
    ds = MNIST('/content/mnist_geometry', train=False, download=True)
    assert int(ds.targets[idx]) == 1, f'MNIST test index {idx} has label {int(ds.targets[idx])}, not 1'
    x = ds.data[idx].float()/127.5 - 1.0
    train = MNIST('/content/mnist_geometry', train=True, download=True)
    return x.reshape(1,1,28,28), train


class ScoreOperator:
    """Symmetric part of normalized score Hessian Q, matrix-free."""
    def __init__(self, net, x, t, fd_eps=0.04):
        self.net = net
        self.t = int(t)
        self.ab = alpha_cumprod(t)
        self.sqrt_ab = math.sqrt(self.ab)
        self.scale = math.sqrt(1-self.ab)
        self.sigma = math.sqrt((1-self.ab)/self.ab)
        self.xt = self.sqrt_ab * x.detach().clone()
        self.device = x.device
        self.D = x.numel()
        self.fd_eps = float(fd_eps)
        self.image_forward_evals = 0
        self.backward_evals = 0
        self.sym_calls = 0
        self.raw_calls = 0
        self.adj_calls = 0

    def model_call(self, batch):
        ids=torch.full((batch.shape[0],),self.t,device=self.device,dtype=torch.long)
        labels=torch.ones(batch.shape[0],device=self.device,dtype=torch.long)
        return self.net(batch,ids,class_labels=labels).sample

    def q_raw(self, vector, h=None):
        """Q_raw v by forward differences of epsilon network (only forwards)."""
        self.raw_calls += 1
        v = vector.detach().to(self.xt).reshape_as(self.xt)
        n = float(torch.linalg.vector_norm(v))
        if n < 1e-15: return torch.zeros_like(vector)
        step = (h if h is not None else self.fd_eps)/n
        with torch.no_grad():
            xs=torch.cat([self.xt+step*v,self.xt-step*v],dim=0)
            out=self.model_call(xs)
            self.image_forward_evals += 2
            jv=(out[0:1]-out[1:2])/(2*step)
        return (self.scale*jv).reshape(-1)

    def q_adj(self, vector):
        """Q_raw^T v by first-order reverse-mode gradient, no double backward."""
        self.adj_calls += 1
        v=vector.detach().to(self.xt).reshape_as(self.xt)
        with torch.enable_grad():
            inp=self.xt.detach().clone().requires_grad_(True)
            out=self.model_call(inp)
            scalar=(out*v).sum()
            grad=torch.autograd.grad(scalar,inp,create_graph=False)[0]
        self.image_forward_evals += 1
        self.backward_evals += 1
        return (self.scale*grad).detach().reshape(-1)

    def q_sym(self, vector):
        self.sym_calls += 1
        return 0.5*(self.q_raw(vector)+self.q_adj(vector))

    def counters(self):
        return dict(sym_calls=self.sym_calls,fd_calls=self.raw_calls,
                    vjp_calls=self.adj_calls,model_forward_image_evals=self.image_forward_evals,
                    model_backward_evals=self.backward_evals)


class DiagOp:
    def __init__(self, diag):
        self.diag=torch.as_tensor(diag,dtype=torch.float32)
        self.D=self.diag.numel()
        self.device=self.diag.device
    def q_sym(self,v): return self.diag * v


def lanczos(op, v, nsteps=12):
    z=v.detach().flatten().float().to(op.device)
    mag=float(torch.linalg.vector_norm(z).item())
    if mag < 1e-14: raise ValueError('Near-zero Lanczos start vector')
    z=z/mag
    basis=[]; alphas=[];betas=[]
    prev=torch.zeros_like(z); prev_beta=0.0
    for it in range(min(nsteps,z.numel())):
        w=op.q_sym(z)
        a=float(torch.dot(z,w).item())
        w=w-a*z-prev_beta*prev
        # Two-pass full reorthogonalization for spectral numerical robustness.
        for _ in range(2):
            for q in basis+[z]:
                w=w-torch.dot(q,w)*q
        b=float(torch.linalg.vector_norm(w))
        basis.append(z)
        alphas.append(a)
        if b < 1e-8 or it==nsteps-1: break
        betas.append(b)
        prev=z;prev_beta=b;z=w/b
    T=np.diag(np.asarray(alphas,dtype=np.float64))
    for i,b in enumerate(betas):
        T[i,i+1]=b; T[i+1,i]=b
    eig,U=np.linalg.eigh(T)
    return eig,U,basis,mag


def hill(eig):
    return 1./(1.+(np.abs(eig)/SELECT['spectral_cutoff'])**SELECT['hill_power'])


def spectral_data(op,steps,probes,seed):
    gen=torch.Generator(device='cpu').manual_seed(seed)
    vals=[]
    for p in range(probes):
        z=(torch.randint(0,2,(op.D,),generator=gen,dtype=torch.int64)*2-1).float().to(op.device)
        lam, U, _, _ = lanczos(op,z,steps)
        weights=U[0,:]**2
        deff=float(op.D*np.dot(weights,hill(lam)))
        transition=float(np.dot(weights,( (np.abs(lam)>=.05)&(np.abs(lam)<=.20)).astype(float)))
        violation=float(np.dot(weights,np.maximum(-lam,0)+np.maximum(lam-1,0)))
        vals.append((deff,transition,violation))
    arr=np.asarray(vals)
    return dict(d_eff=float(arr[:,0].mean()), d_round=int(np.rint(arr[:,0].mean())),
                d_probe_SE=float(arr[:,0].std(ddof=1)/math.sqrt(probes)) if probes>=2 else np.nan,
                transition=float(arr[:,1].mean()),range_violation=float(arr[:,2].mean()),
                lanczos_calls_max=steps*probes)


def probe_vectors(D,seed,count,device):
    gen=torch.Generator(device='cpu').manual_seed(seed)
    out=[]
    for _ in range(count):
        z=(torch.randint(0,2,(D,),generator=gen)*2-1).float().to(device)
        out.append(z/math.sqrt(D))
    return out


def geometry_data(ops,probes=2,seed=123):
    # Common probes across ALL scales: reduces comparison noise.
    zlist=probe_vectors(ops[0].D,seed,probes,ops[0].device)
    geo=[]
    for i,op in enumerate(ops):
        numerator=denom=asym_num=asym_den=cnum=cden=0.0
        adj_ids=[j for j in [i-1,i+1] if 0<=j<len(ops)]
        for z in zlist:
            q=op.q_sym(z);qq=op.q_sym(q)
            numerator += float(torch.sum((qq-q)**2))
            denom += float(torch.sum(q**2))
            raw=op.q_raw(z);adj=op.q_adj(z)
            asym_num += float(torch.sum((raw-adj)**2))
            asym_den += .5*float(torch.sum(raw**2)+torch.sum(adj**2))
            for j in adj_ids:
                qr=ops[j].q_sym(z)
                cnum += float(torch.sum((q-qr)**2))
                cden += float(torch.sum(q**2))
        geo.append(dict(projector_defect=math.sqrt(numerator/max(denom,1e-20)),
                        antisymmetry=math.sqrt(asym_num/max(asym_den,1e-20)),
                        cross_scale=math.sqrt(cnum/max(cden,1e-20))))
    return geo


def scale_selection(rows,method):
    # Exact same threshold and contiguous-region rules as previous bridge.
    rows=sorted(rows,key=lambda r:r['sigma'])
    for r in rows:
        r['spectral_valid']=bool(r['transition']<=SELECT['transition'] and 0.5<r['d_eff']<784-0.5)
        r['joint_valid']=bool(r['spectral_valid'] and r['projector_defect']<=SELECT['defect'] and
            r['cross_scale']<=SELECT['cross'] and r['antisymmetry']<=SELECT['asym'] and
            r['range_violation']<=SELECT['range'])
    key='spectral_valid' if method=='LHSD_inspired_transition' else 'joint_valid'
    groups=[];group=[]
    for i,r in enumerate(rows):
        if r[key]: group.append(i)
        elif group: groups.append(group);group=[]
    if group:groups.append(group)
    groups=[g for g in groups if len(g)>=SELECT['min_run']]
    if not groups:return None
    if method=='LHSD_inspired_transition':
        g=min(groups,key=lambda ids:(-len(ids),np.mean([rows[j]['transition'] for j in ids])))
        return rows[g[(len(g)-1)//2]]
    metric=lambda r:r['transition']+r['projector_defect']+r['cross_scale']
    g=min(groups,key=lambda ids:(np.mean([metric(rows[j]) for j in ids]),-len(ids)))
    return rows[min(g,key=lambda j:metric(rows[j]))]


def filter_apply(op, vector, steps):
    v=vector.detach().flatten().float().to(op.device)
    eig,U,basis,mag=lanczos(op,v,steps)
    coeff=(U * (hill(eig)*U[0,:])[None,:]).sum(axis=1)
    out=sum(float(c)*q for c,q in zip(coeff,basis))
    return mag*out


def shifted_image(x, dx, dy):
    # Subpixel shifts (not used for score selection).
    theta=torch.tensor([[[1.,0.,-2*dx/28],[0.,1.,-2*dy/28]]],dtype=x.dtype,device=x.device)
    grid=F.affine_grid(theta,x.size(),align_corners=False)
    return F.grid_sample(x,grid,mode='bilinear',padding_mode='border',align_corners=False)


def independent_vectors(x,train,neighbors=60):
    # Never run before scale selection; independent retrospective evaluation only.
    h=.25
    vx=((shifted_image(x,h,0)-shifted_image(x,-h,0))/(2*h)).flatten()
    vy=((shifted_image(x,0,h)-shifted_image(x,0,-h))/(2*h)).flatten()
    alltrain=train.data[train.targets==1].float().reshape(-1,784)/127.5-1.0
    xcpu=x.cpu().flatten()
    dists=torch.linalg.vector_norm(alltrain-xcpu.unsqueeze(0),dim=1)
    nearest=torch.topk(dists,k=neighbors,largest=False).indices
    delta=(alltrain[nearest]-xcpu.unsqueeze(0))
    _,_,vt=torch.linalg.svd(delta,full_matrices=False)
    vecs={'translation_x':vx,'translation_y':vy}
    for i in range(4):vecs[f'neighbor_pc_{i+1}']=vt[i].to(x.device)
    gen=torch.Generator(device='cpu').manual_seed(90210)
    for i in range(4):vecs[f'random_{i+1}']=torch.randn(784,generator=gen).to(x.device)
    return vecs


def independent_validation(op, vectors, steps, method):
    rows=[]
    for name,vec in vectors.items():
        base=vec.flatten().float().to(op.device)
        v=base/torch.linalg.vector_norm(base).clamp_min(1e-12)
        try:
            projected=filter_apply(op,v,steps)
            retention=float(torch.sum(projected**2))
            reconstruction_error=float(torch.linalg.vector_norm(projected-v))
            cosine=float(torch.dot(projected,v)/torch.linalg.vector_norm(projected).clamp_min(1e-12))
        except Exception as e:
            raise RuntimeError(f'Failed independent validation {name}') from e
        rows.append(dict(method=method,direction=name,squared_retention=retention,
                         reconstruction_error=reconstruction_error,cosine=cosine,
                         vector_norm_original=float(torch.linalg.vector_norm(base))))
    return rows


def derivative_audit(op,seed=7):
    z=probe_vectors(op.D,seed,1,op.device)[0]
    j1=op.q_raw(z,h=0.04); j2=op.q_raw(z,h=0.02)
    rel=float(torch.linalg.vector_norm(j1-j2)/torch.linalg.vector_norm(j2).clamp_min(1e-10))
    # Scalar reciprocity: <u, Jv> = <J^T u, v>.
    u=probe_vectors(op.D,seed+1,1,op.device)[0]
    raw=op.q_raw(z);adj=op.q_adj(u)
    pair=float(abs(torch.dot(u,raw)-torch.dot(adj,z)) /
               max(abs(float(torch.dot(u,raw))), abs(float(torch.dot(adj,z))),1e-6))
    return dict(fd_step_relative_disagreement=rel,adjoint_identity_relative_error=pair)


def plot_results(out,rows,sel,x,ind):
    import matplotlib.pyplot as plt
    d=pd.DataFrame(rows).sort_values('sigma')
    fig,axes=plt.subplots(2,3,figsize=(14,8))
    for ax,k in zip(axes.flat,['d_eff','transition','projector_defect','cross_scale','antisymmetry','range_violation']):
        ax.plot(d['sigma'],d[k],marker='o')
        for s in sel:
            if s['accepted']:
                ax.axvline(s['sigma'],linestyle='--',label=s['method'])
        ax.set_xscale('log');ax.set_title(k);ax.set_xlabel('Effective VE sigma')
        ax.grid(alpha=.3)
    axes.flat[0].legend(fontsize=7)
    fig.suptitle('One held-out MNIST digit 1 | same checkpoint & noise scales')
    fig.tight_layout();fig.savefig(out/'scale_diagnostics.png',dpi=145);plt.close(fig)
    fig,axs=plt.subplots(1,2,figsize=(10,4))
    axs[0].imshow(x.detach().cpu().reshape(28,28),cmap='gray',vmin=-1,vmax=1)
    axs[0].set_title('Held-out real MNIST digit 1');axs[0].axis('off')
    d2=pd.DataFrame(ind)
    if not d2.empty:
        d2=d2.pivot(index='direction',columns='method',values='squared_retention')
        d2.plot(kind='bar',ax=axs[1],rot=45)
        axs[1].set_ylabel('Squared tangent-filter retention')
        axs[1].set_title('Independent direction checks')
    else:axs[1].text(.1,.4,'Both methods abstained\nNo geometry claim',fontsize=12)
    fig.tight_layout();fig.savefig(out/'independent_geometry.png',dpi=145);plt.close(fig)


def make_report(out, rows, selections, independent, audit, cfg, idx, device):
    df=pd.DataFrame(independent)
    lines=['# MNIST digit-1 score-geometry feasibility report','',
           f'- Model: `{MODEL_ID}` (frozen pretrained, class label 1).',
           f'- Held-out MNIST test index: {idx}; input dimension 784; model inference device: {device}.',
           '- Score Jacobian: symmetric part of epsilon-prediction Jacobian; central finite difference + first-order reverse mode.',
           '- Comparator: LHSD-**inspired** transition selector, **NOT official LHSD code**.',
           '- Selector rules and thresholds frozen from earlier synthetic benchmark; translations and neighbor data never used for selection.',
           '- MNIST true intrinsic dimension is **unknown**. A dimension estimate is not an accuracy measurement.','',
           '## Scale selection (unverified dimension values unless independently validated)','',
           '| Method | Status | DDPM timestep | VE-equivalent sigma | Estimated ID (SLQ) | SLQ probe SE |',
           '|---|---|---:|---:|---:|---:|']
    for s in selections:
        lines.append(f"| {s['method']} | {'ACCEPT' if s['accepted'] else 'ABSTAIN'} | " +
                     (f"{s['timestep']} | {s['sigma']:.5f} | {s['d_eff']:.2f} | {s['d_probe_SE']:.2f} |" if s['accepted'] else '— | — | — | — |'))
    lines+=['','## Independent checks','']
    if not df.empty:
        for m,g in df.groupby('method'):
            tr=g[g.direction.str.startswith('translation')]['squared_retention'].mean()
            rand=g[g.direction.str.startswith('random')]['squared_retention'].mean()
            nn=g[g.direction.str.startswith('neighbor')]['squared_retention'].mean()
            lines.append(f'- {m}: translation retention {tr:.4f}; random retention {rand:.4f}; clean-neighbor PC retention {nn:.4f}.')
        lines.append('- Retention is a **heuristic** based on a soft spectral tangent filter and is influenced by estimated dimension; do not interpret as true tangent error.')
    else:lines.append('- Both selectors abstained: no independently tested tangent estimate.')
    lines += ['', '## Numerical correctness (at the middle noise scale)',
              f"- Forward-difference step disagreement: {audit['fd_step_relative_disagreement']:.5g}.",
              f"- First-order adjoint duality residual: {audit['adjoint_identity_relative_error']:.5g}.",
              '- Numerical derivative check: '+('PASS (heuristic 5% tolerance).' if max(audit.values()) < 0.05 else '**WARNING: failed 5% tolerance; do not rely on estimated geometry.**'),
              '- Small residuals support the numerical operator only; they do not establish that the pretrained network has learned manifold geometry.',
              '', '## Scientific interpretation',
              '- If one method has better independently measured retention **at a comparable estimated dimension**, this is preliminary real-image geometry evidence, not MNIST true-ID accuracy.',
              '- If both abstain or both align poorly, report that this checkpoint does not support a credible geometry recovery demonstration at the tested scales.',
              '- More images, trained known-dimension score models, official LHSD replication, finite-difference and SLQ convergence are future work.','']
    (out/'PROPOSAL_REPORT.md').write_text('\n'.join(lines))


def run(args):
    import matplotlib
    matplotlib.use('Agg')
    torch.manual_seed(19);np.random.seed(19)
    torch.set_num_threads(min(4,os.cpu_count() or 2))
    cfg=config_for(args.mode)
    device=args.device if args.device != 'auto' else ('cuda' if torch.cuda.is_available() else 'cpu')
    out=Path(args.out);out.mkdir(parents=True,exist_ok=True)
    x,train=image_load(args.index)
    x=x.float().to(device)
    net=model_load(device)
    ops=[ScoreOperator(net,x,t,cfg['fd_eps']) for t in cfg['timesteps']]
    ops.sort(key=lambda a:a.sigma)
    with torch.no_grad():
        check=ops[0].model_call(ops[0].xt)
        assert check.shape==ops[0].xt.shape and torch.isfinite(check).all(), 'Pretrained conditional model failed inference sanity check'
    rows=[]
    print('Pretrained MNIST-conditional checkpoint loaded; clean test label 1;',len(ops),'scales',flush=True)
    for op in ops:
        tick=time.perf_counter()
        stats=spectral_data(op,cfg['lanczos_steps'],cfg['trace_probes'],seed=7)
        rows.append(dict(timestep=op.t,sigma=op.sigma,**stats,slq_seconds=time.perf_counter()-tick))
        print(f"  t={op.t:3d} sigma={op.sigma:.4f} ID(SLQ)={stats['d_eff']:.2f} transition={stats['transition']:.3f}",flush=True)
    print('Computing projector/cross-scale diagnostics...',flush=True)
    geo=geometry_data(ops,cfg['geom_probes'])
    for row,g in zip(rows,geo): row.update(g)
    selections=[]
    for name in ['LHSD_inspired_transition','Projector_cross_scale']:
        choice=scale_selection(rows,name)
        if choice is None:
            selections.append(dict(method=name,accepted=False,timestep=np.nan,sigma=np.nan,d_eff=np.nan,d_probe_SE=np.nan,reason='No consecutive accepted noise-scale interval'))
        else:
            selections.append(dict(method=name,accepted=True,timestep=choice['timestep'],sigma=choice['sigma'],d_eff=choice['d_eff'],d_probe_SE=choice['d_probe_SE'],reason='Frozen selector passed'))
    print('Frozen selectors:',[(s['method'], 'ABSTAIN' if not s['accepted'] else (s['timestep'],round(s['d_eff'],2))) for s in selections],flush=True)
    mid=ops[len(ops)//2]
    print('Running numerical finite-difference/adjoint audit...',flush=True)
    audit=derivative_audit(mid)
    # Held-out translations/clean-neighbor data created AFTER selection. Cannot affect the decision.
    ind=[]
    if any(s['accepted'] for s in selections):
        vectors=independent_vectors(x,train,neighbors=cfg['ref_neighbors'])
        for s in selections:
            if s['accepted']:
                op=next(o for o in ops if o.t == s['timestep'])
                print(f"Independent validation: {s['method']} at t={op.t}...",flush=True)
                ind.extend([dict(timestep=op.t,sigma=op.sigma,**r) for r in
                            independent_validation(op,vectors,cfg['filter_steps'],s['method'])])
    for row,op in zip(rows,ops):row.update(op.counters())
    df=pd.DataFrame(rows);sd=pd.DataFrame(selections);id_df=pd.DataFrame(ind)
    df.to_csv(out/'all_scale_diagnostics.csv',index=False)
    sd.to_csv(out/'selected_scales.csv',index=False)
    id_df.to_csv(out/'independent_tangent_checks.csv',index=False)
    (out/'numerical_audit.json').write_text(json.dumps(audit,indent=2))
    meta=dict(checkpoint=MODEL_ID,checkpoint_architecture='UNet2DModel num_class_embeds=11',
              test_index=args.index,test_label=1,input_scaling='uint8/127.5-1',
              input_sha256=hashlib.sha256(x.detach().cpu().numpy().tobytes()).hexdigest(),
              class_label_used_by_model=1,noise_schedule='DDPM linear beta 0.0001 to 0.02, T=1000',
              Q='sqrt(1-alpha_bar)*sym(J_xt epsilon_theta(xt,t,label=1))',
              model_evaluated_at='xt=sqrt(alpha_bar)*clean_x (no random test corruption)',
              noise_scale='VE-equivalent sigma=sqrt((1-alpha_bar)/alpha_bar)',
              fd_step=cfg['fd_eps'],selection_thresholds=SELECT,config=cfg,device=device,
              official_LHSD=False,ground_truth_MNIST_dimension=None,
              independent_data_used_only_after_selection=True)
    (out/'protocol.json').write_text(json.dumps(meta,indent=2))
    torch.save(dict(x_clean=x.detach().cpu(), test_index=args.index, model_id=MODEL_ID,
                    selected_timesteps={s['method']:(int(s['timestep']) if s['accepted'] else None) for s in selections},
                    note='Reusable pixel-space input; reload the same pretrained score model and ScoreOperator before curvature/projection.'),
               out/'geometry_state.pt')
    plot_results(out,rows,selections,x,ind)
    make_report(out,rows,selections,ind,audit,cfg,args.index,device)
    print('\nSELECTED SCALES:\n'+sd.to_string(index=False))
    print('\nNumerical audit:',audit)
    print('\nFiles saved:',out.resolve())
    print('Read PROPOSAL_REPORT.md. No real-MNIST dimension ground truth. Not official LHSD.')


class LinearEpsMock(torch.nn.Module):
    """A model used only in the unit test, never as MNIST results."""
    def __init__(self, diag):
        super().__init__()
        self.register_buffer('diag',torch.tensor(diag,dtype=torch.float32).reshape(1,1,28,28))
    def forward(self,x,t,class_labels):
        return SimpleNamespace(sample=x*self.diag)


def self_test():
    print('Running analytic operator, SLQ and soft-filter unit tests (no pretrained checkpoint).')
    t=35; scale=math.sqrt(1-alpha_cumprod(t))
    vals=np.concatenate([np.zeros(120),np.ones(784-120)]).astype(np.float32)
    net=LinearEpsMock(vals/scale)
    x=torch.zeros(1,1,28,28)
    op=ScoreOperator(net,x,t)
    g=torch.Generator().manual_seed(42)
    v=torch.randn(784,generator=g)
    assert torch.allclose(op.q_raw(v),torch.as_tensor(vals)*v,atol=1e-3,rtol=1e-3)
    assert torch.allclose(op.q_adj(v),torch.as_tensor(vals)*v,atol=1e-5,rtol=1e-5)
    d=spectral_data(op,steps=8,probes=6,seed=42)
    # Check tridiagonal reconstruction on known vector and exact diagonal matrix
    dv=DiagOp(vals)
    u=torch.randn(784,generator=g)
    result=filter_apply(dv,u,steps=8)
    expected=torch.as_tensor(hill(vals))*u
    assert torch.allclose(result,expected,atol=3e-4,rtol=3e-4),float(torch.linalg.vector_norm(result-expected))
    print('PASS Q finite difference, adjoint first derivative, Lanczos filter. Estimated analytic soft ID:',round(d['d_eff'],2))


def main():
    p=argparse.ArgumentParser()
    p.add_argument('--mode',choices=['pilot','expanded'],default='pilot')
    p.add_argument('--out',default='/content/mnist_digit1_final_results')
    p.add_argument('--index',type=int,default=2,help='Held-out MNIST digit-1 test index; default 2')
    p.add_argument('--device',choices=['auto','cpu','cuda'],default='auto')
    p.add_argument('--self-test',action='store_true')
    a=p.parse_args()
    if a.self_test:self_test()
    else:run(a)


if __name__=='__main__': main()


In [ ]:
%%writefile /content/MNIST_Original_vs_Denoised.py
#!/usr/bin/env python3
"""Compare frozen geometry estimator at original vs DDPM-denoised real MNIST digit 1.

Standalone if placed next to MNIST_Digit1_Final_Geometry.py (bundled and embedded
into the accompanying standalone Colab). Both original & once-denoised anchors
are evaluated with the SAME pretrained network, same sigma grid, same probes,
and identical frozen selection criteria. All independent validation is post hoc.

This uses the predicted-x0 formula for an epsilon-predicting VP DDPM, with
x_t = sqrt(alpha_bar)*x_clean (zero random noise). This is a deterministic
score-based update, not a mathematically certified manifold projection.
"""
import argparse
import json
import math
import os
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import MNIST_Digit1_Final_Geometry as geom


def denoise_once(net, x, timestep):
    """VP predicted-x0 at a precommitted timestep; NO pixel clipping."""
    ab = geom.alpha_cumprod(timestep)
    xt = math.sqrt(ab) * x
    ts = torch.tensor([int(timestep)], device=x.device, dtype=torch.long)
    labels = torch.ones(1, device=x.device, dtype=torch.long)
    with torch.no_grad():
        predicted_epsilon = net(xt, ts, class_labels=labels).sample
        xhat = (xt - math.sqrt(1 - ab) * predicted_epsilon) / math.sqrt(ab)
    assert torch.isfinite(xhat).all() and xhat.shape == x.shape
    return xhat, dict(denoise_timestep=timestep,
         denoise_sigma=math.sqrt((1-ab)/ab),
         image_l2=float(torch.linalg.vector_norm(xhat-x)),
         image_rms=float(torch.mean((xhat-x)**2).sqrt()),
         image_relative_l2=float(torch.linalg.vector_norm(xhat-x)/torch.linalg.vector_norm(x).clamp_min(1e-9)),
         fraction_outside_input_range=float(((xhat < -1.) | (xhat > 1.)).float().mean()),
         image_max_abs_change=float((xhat-x).abs().max()),
         fraction_foreground_changed=float(((xhat > 0) != (x > 0)).float().mean()))


def compute_anchor(label, net, x, cfg):
    ops = sorted([geom.ScoreOperator(net, x, t, cfg['fd_eps']) for t in cfg['timesteps']],key=lambda op:op.sigma)
    records=[]
    for op in ops:
        t0=time.perf_counter()
        results=geom.spectral_data(op,cfg['lanczos_steps'],cfg['trace_probes'],seed=7)
        records.append(dict(anchor=label,timestep=op.t,sigma=op.sigma,**results,
                            spectral_seconds=time.perf_counter()-t0))
        print(f"{label:9s} t={op.t:3d}: sigma={op.sigma:.4f}, ID={results['d_eff']:.2f}, "
              f"transition={results['transition']:.4f}",flush=True)
    geo=geom.geometry_data(ops,cfg['geom_probes'],seed=123)
    for rec,g in zip(records,geo): rec.update(g)
    choices=[]
    for method in ['LHSD_inspired_transition','Projector_cross_scale']:
        found=geom.scale_selection(records,method)
        if found is None:
            choices.append(dict(anchor=label,method=method,accepted=False,
                 timestep=np.nan,sigma=np.nan,d_eff=np.nan,d_probe_SE=np.nan,
                 reason='No consecutive scales passed frozen criteria'))
        else:
            choices.append(dict(anchor=label,method=method,accepted=True,
                 timestep=found['timestep'],sigma=found['sigma'],
                 d_eff=found['d_eff'],d_probe_SE=found['d_probe_SE'],
                 reason='Frozen selector accepted'))
    audit=geom.derivative_audit(ops[len(ops)//2])
    for op,rec in zip(ops,records):rec.update(op.counters())
    return ops, records, choices, audit


def independent_comparison(anchors,ops_by_anchor, choices, train, cfg):
    # Same independent reference directions for both anchors, defined at clean x.
    original=anchors['original']
    vectors=geom.independent_vectors(original,train,neighbors=cfg['ref_neighbors'])
    out=[]
    for label in ['original','denoised']:
        ops=ops_by_anchor[label]
        todo={cfg['comparison_timestep']:['fixed_timestep_diagnostic']}
        for c in choices:
            if c['anchor']==label and c['accepted']:
                todo.setdefault(int(c['timestep']),[]).append(c['method'])
        for t,roles in sorted(todo.items()):
            op=next(o for o in ops if o.t==t)
            print(f"Independent validation {label} t={t}, roles={roles}",flush=True)
            res=geom.independent_validation(op,vectors,cfg['filter_steps'],method='posthoc')
            for row in res:
                out.append(dict(anchor=label,timestep=t,sigma=op.sigma,roles=';'.join(roles),**row))
    return out


def aggregate_checks(vals):
    df=pd.DataFrame(vals)
    if df.empty:return pd.DataFrame()
    df['group']=df['direction'].map(lambda s:'translation' if s.startswith('translation')
                   else ('neighbors' if s.startswith('neighbor') else 'random'))
    f=(df.groupby(['anchor','timestep','group'])[['squared_retention','reconstruction_error']].mean().reset_index())
    pivot=f.pivot(index=['anchor','timestep'],columns='group',values='squared_retention').reset_index()
    pivot.columns.name=None
    if 'random' in pivot.columns:
        for k in ['translation','neighbors']:
            if k in pivot.columns:pivot[k+'_minus_random']=pivot[k]-pivot['random']
    return pivot


def make_plots(out, x, xhat, records, agg, timestep):
    orig=x.detach().cpu().reshape(28,28).numpy()
    den=xhat.detach().cpu().reshape(28,28).numpy()
    fig,axes=plt.subplots(1,3,figsize=(10,3))
    for ax,img,title in zip(axes,[orig,den,np.abs(orig-den)],
                            ['Original clean digit 1','Denoised (unclipped)','Absolute change']):
        kwargs=dict(cmap='gray') if 'change' in title.lower() else dict(cmap='gray',vmin=-1,vmax=1)
        im=ax.imshow(img,**kwargs);ax.set_title(title);ax.axis('off')
    fig.tight_layout();fig.savefig(out/'original_denoised_images.png',dpi=150);plt.close(fig)
    df=pd.DataFrame(records)
    keys=['d_eff','projector_defect','range_violation','cross_scale','antisymmetry','transition']
    fig,ax=plt.subplots(2,3,figsize=(14,8))
    for a,k in zip(ax.flat,keys):
        for label,grp in df.groupby('anchor'):
            grp=grp.sort_values('sigma')
            a.plot(grp['sigma'],grp[k],marker='o',label=label)
        a.set_title(k);a.set_xlabel('VE-equivalent sigma');a.set_xscale('log');a.grid(alpha=.25)
        threshold={'projector_defect':geom.SELECT['defect'], 'range_violation':geom.SELECT['range'],
                   'cross_scale':geom.SELECT['cross'], 'antisymmetry':geom.SELECT['asym'],
                   'transition':geom.SELECT['transition']}.get(k)
        if threshold is not None:a.axhline(threshold,color='k',lw=1,ls='--',alpha=.55)
    ax.flat[0].legend()
    fig.suptitle(f'Original vs denoised | denoise at t={timestep}; identical scale grids')
    fig.tight_layout();fig.savefig(out/'geometry_comparison.png',dpi=160);plt.close(fig)
    if not agg.empty:
        d=agg[agg['timestep']==timestep].set_index('anchor')
        categories=[k for k in ['translation','neighbors','random'] if k in d.columns]
        if len(d)==2 and categories:
            fig,ax=plt.subplots(figsize=(8,4))
            d.loc[['original','denoised'],categories].T.plot.bar(ax=ax,rot=0)
            ax.set_ylabel('Mean squared tangent-filter response');ax.set_ylim(bottom=0)
            ax.set_title(f'Independent direction checks at same timestep {timestep}')
            fig.tight_layout();fig.savefig(out/'independent_geometry_comparison.png',dpi=160);plt.close(fig)


def report(out, records, selections, changes, independent_summary, audit, cfg):
    df=pd.DataFrame(records)
    same=df[df.timestep==cfg['comparison_timestep']].set_index('anchor')
    s=pd.DataFrame(selections)
    lines=['# Original vs denoised: real MNIST digit `1` geometric comparison','',
           'This is an **exploratory real-image feasibility comparison**, NOT true MNIST dimension accuracy.','',
           f"- Frozen pretrained checkpoint: `{geom.MODEL_ID}`, class condition 1.",
           f"- Denoising timestep chosen before this A/B run: **{cfg['comparison_timestep']}**.",
           '- Single deterministic VP predicted-x0 correction evaluated from clean image (no sampled Gaussian noise).',
           '- Denoised image is **not clipped** for geometry; no claimed true manifold projection.',
           '- Identical model, noise grid, estimator and random probes for both anchors.',
           '- Independent translation and clean-neighbor checks are **never used to select noise scales**.',
           '- Thresholds unchanged from earlier synthetic evaluation; selection is exploratory only.','',
           '## Image shift',
           f"- RMS image change: **{changes['image_rms']:.5f}** on [-1,1] image scale.",
           f"- Relative L2 change: **{changes['image_relative_l2']:.3f}**.",
           f"- Denoised pixels outside [-1,1]: **{changes['fraction_outside_input_range']*100:.2f}%**.",
           f"- Pixels changing sign (zero-threshold proxy): **{changes['fraction_foreground_changed']*100:.2f}%**.",'',
           '## Matched-timestep diagnostics (no selection bias)','',
           f"Both operators are evaluated at t={cfg['comparison_timestep']}:",'',
           '| Metric | Original | Denoised | Delta (denoised minus original) |','|---|---:|---:|---:|']
    for k in ['d_eff','d_probe_SE','transition','projector_defect','range_violation','cross_scale','antisymmetry']:
        a=float(same.loc['original',k]);b=float(same.loc['denoised',k])
        lines.append(f'| {k} | {a:.5f} | {b:.5f} | {b-a:+.5f} |')
    lines+=['','## Frozen selector outcomes','',
            '| Anchor | Selector | Status | Sigma | Estimated dimension |','|---|---|---|---:|---:|']
    for row in selections:
        lines.append(f"| {row['anchor']} | {row['method']} | {'ACCEPT' if row['accepted'] else 'ABSTAIN'} | "
                     +(f"{row['sigma']:.5f} | {row['d_eff']:.2f} |" if row['accepted'] else '— | — |'))
    lines+=['','## Retrospective independent direction checks (fixed t)','']
    if not independent_summary.empty:
        mm=independent_summary[independent_summary.timestep==cfg['comparison_timestep']]
        for _, r in mm.iterrows():
            lines.append(f"- {r['anchor']}: translation={r.get('translation',float('nan')):.4f}, "
                         f"neighbors={r.get('neighbors',float('nan')):.4f}, "
                         f"random={r.get('random',float('nan')):.4f}, "
                         f"translation−random={r.get('translation_minus_random',float('nan')):+.4f}, "
                         f"neighbors−random={r.get('neighbors_minus_random',float('nan')):+.4f}.")
    lines+=['','**Important:** Lower defect alone is NOT evidence of better tangent geometry. A higher '
            'semantic-minus-random response is evidence of selectivity, but its interpretation is still '
            'confounded when the estimated tangent dimension changes. These checks are exploratory.',
            'Independent data directions and shifts approximate semantic variations; they are not guaranteed '
            'ground-truth tangent vectors of a smooth MNIST manifold.', '',
            '## Numerical derivative audit','']
    for label, data in audit.items():
        lines.append(f"- {label}: finite-difference discrepancy={data['fd_step_relative_disagreement']:.5g}, "
            f"adjoint identity discrepancy={data['adjoint_identity_relative_error']:.5g}.")
    lines+=['','## Interpretation rule','',
            '1. If the denoised point improves projector geometry but **not** independently evaluated '
            'tangent selectivity, it only improves an internal algebraic diagnostic.',
            '2. If the denoised point improves both, report a one-image *feasibility observation*, '
            'not proven intrinsic-dimension accuracy or a certified projection.',
            '3. If denoising changes the digit substantially or causes large out-of-range pixels, '
            'avoid treating it as a nearby manifold point.',
            '4. Never tune thresholds using this image and then present it as held-out confirmation.',
            '5. Neither result is official LHSD. Expand to a separately held-out test set for claims of generality.','']
    (out/'PROPOSAL_REPORT.md').write_text('\n'.join(lines),encoding='utf-8')


def run(args):
    torch.manual_seed(19);np.random.seed(19)
    torch.set_num_threads(min(4,os.cpu_count() or 2))
    cfg=geom.config_for(args.mode)
    if args.timestep not in cfg['timesteps']:
        raise ValueError(f"--timestep {args.timestep} must be included in {cfg['timesteps']}")
    cfg['comparison_timestep']=args.timestep
    device=args.device if args.device!='auto' else ('cuda' if torch.cuda.is_available() else 'cpu')
    out=Path(args.out);out.mkdir(parents=True,exist_ok=True)
    x,train=geom.image_load(args.index)
    x=x.float().to(device)
    net=geom.model_load(device)
    xhat,changes=denoise_once(net,x,args.timestep)
    anchors={'original':x,'denoised':xhat.detach()}
    ops_dict={};all_records=[];all_choices=[];all_audit={}
    for label,a in anchors.items():
        print(f'Running frozen estimator for {label} anchor',flush=True)
        ops,records,choices,audit=compute_anchor(label,net,a,cfg)
        ops_dict[label]=ops;all_records.extend(records)
        all_choices.extend(choices);all_audit[label]=audit
    print('Running retrospective independent checks; these do not affect scale selection...',flush=True)
    ind=independent_comparison(anchors,ops_dict,all_choices,train,cfg)
    agg=aggregate_checks(ind)
    pd.DataFrame(all_records).to_csv(out/'all_scale_diagnostics.csv',index=False)
    pd.DataFrame(all_choices).to_csv(out/'selected_scales.csv',index=False)
    pd.DataFrame(ind).to_csv(out/'independent_tangent_checks.csv',index=False)
    agg.to_csv(out/'independent_summary.csv',index=False)
    (out/'image_change_metrics.json').write_text(json.dumps(changes,indent=2))
    (out/'numerical_audits.json').write_text(json.dumps(all_audit,indent=2))
    (out/'protocol.json').write_text(json.dumps(dict(checkpoint=geom.MODEL_ID,
         anchor_test_index=args.index, conditioning_label=1,
         noise_grid=cfg['timesteps'], denoise_timestep=args.timestep,
         denoise_formula='xhat=(sqrt(ab)*x-sqrt(1-ab)*eps_theta(sqrt(ab)*x,t))/sqrt(ab)',
         no_clipping_for_geometry=True, frozen_thresholds=geom.SELECT,
         same_noise_grid_for_both_anchors=True, mode=args.mode,
         true_MNIST_manifold_dimension=None,
         official_LHSD=False),indent=2))
    torch.save(dict(x_original=x.detach().cpu(),x_denoised=xhat.detach().cpu(),
                    test_index=args.index,denoise_timestep=args.timestep),out/'two_anchor_state.pt')
    make_plots(out,x,xhat,all_records,agg,args.timestep)
    report(out,all_records,all_choices,changes,agg,all_audit,cfg)
    print('\nFROZEN SELECTION DECISIONS:\n'+pd.DataFrame(all_choices).to_string(index=False),flush=True)
    fixed=pd.DataFrame(all_records)
    fixed=fixed[fixed.timestep==args.timestep]
    print('\nFIXED-TIMESTEP GEOMETRY:\n'+fixed[['anchor','d_eff','projector_defect','range_violation','cross_scale','antisymmetry']].to_string(index=False),flush=True)
    print('\nRESEARCH REPORT:',out/'PROPOSAL_REPORT.md',flush=True)
    print('Outputs:',out.resolve(),flush=True)


def self_test():
    class SimpleMock(torch.nn.Module):
        def forward(self, x, t, class_labels):
            # Linear, differentiable, deterministic, no fused attention.
            from types import SimpleNamespace
            return SimpleNamespace(sample=x * 0.15)
    model=SimpleMock()
    x=torch.randn(1,1,28,28).clamp(-1,1)
    xhat,info=denoise_once(model,x,35)
    a=geom.alpha_cumprod(35)
    # Direct expression: xt=sqrt(ab)x, xhat=xt/sqrt(ab) - sqrt((1-ab)/ab) * 0.15*xt
    expected=x - math.sqrt((1-a)/a)*0.15*math.sqrt(a)*x
    assert torch.allclose(xhat,expected,atol=1e-6),float((xhat-expected).abs().max())
    op=geom.ScoreOperator(model,x,35)
    v=torch.randn(784)
    got=op.q_sym(v)
    want=(math.sqrt(1-a)*0.15)*v
    assert torch.allclose(got,want,atol=1e-3,rtol=1e-3),float((got-want).abs().max())
    assert info['image_l2']>0
    print('PASS: DDPM denoiser formula and unchanged score Jacobian Q. No checkpoint downloaded.')


def main():
    p=argparse.ArgumentParser()
    p.add_argument('--mode',choices=['pilot','expanded'],default='pilot')
    p.add_argument('--index',type=int,default=2)
    p.add_argument('--timestep',type=int,default=35)
    p.add_argument('--out',default='/content/mnist_original_vs_denoised')
    p.add_argument('--device',choices=['auto','cpu','cuda'],default='auto')
    p.add_argument('--self-test',action='store_true')
    args=p.parse_args()
    if args.self_test:self_test()
    else:run(args)

if __name__=='__main__': main()


In [ ]:
%%writefile /content/MNIST_Reliability_Calibration_Addon.py
#!/usr/bin/env python3
"""Held-out calibration of finite-noise geometry reliability, for the
MNIST_Original_vs_Denoised_Colab.ipynb experiment.

Calibrates observable *translation and held-out digit-1 neighbor tangent
retention* as a proxy. Does NOT calibrate true intrinsic-dimension error.
Uses distinct MNIST test images for threshold construction, risk calibration,
and the original final demonstration. No threshold adjustment on target.

Prerequisites: MNIST_Digit1_Final_Geometry.py and
MNIST_Original_vs_Denoised.py in this script's directory (/content).
"""
import argparse
import json
import math
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.stats import norm
from matplotlib import pyplot as plt

import MNIST_Digit1_Final_Geometry as geom
from MNIST_Original_vs_Denoised import denoise_once

# IMPORTANT: established *before* inspecting calibration or target outcomes.
QUANTILES=(0.50, 0.75, 0.90, 1.00)
PROXY_WEIGHT_TRANSLATION = 0.5
PROXY_WEIGHT_NEIGHBORS = 0.5
CONFIDENCE_DELTA = 0.10
MAX_PROXY_RISK = 0.65


def get_test_images(target_index, n_fit, n_cal):
    from torchvision.datasets import MNIST
    ds=MNIST('/content/mnist_geometry',train=False,download=True)
    assert int(ds.targets[target_index])==1, 'The target index must label digit 1.'
    indices=[int(i) for i in torch.where(ds.targets==1)[0].tolist() if int(i)!=target_index]
    # Deterministic split; no target image in the calibration sample.
    rng=np.random.default_rng(7604)
    rng.shuffle(indices)
    selected=indices[:n_fit+n_cal]
    if len(selected)!=n_fit+n_cal: raise RuntimeError('Not enough separate digit-1 test images.')
    return ds,selected[:n_fit],selected[n_fit:]


def image_tensor(ds,idx,device):
    return (ds.data[idx].to(device).float()/127.5-1).reshape(1,1,28,28)


def heldout_directions(x, ds, source_idx, device, excluded_indices, n_neighbors=40):
    """Semantic proxy directions from the untrained-on test partition.
    Includes translation tangents and nearest test-set digit-1 local PCA;
    random controls have the same dimensionality and are not fitted.
    """
    h=0.25
    vx=(geom.shifted_image(x,h,0)-geom.shifted_image(x,-h,0)).flatten()/(2*h)
    vy=(geom.shifted_image(x,0,h)-geom.shifted_image(x,0,-h)).flatten()/(2*h)
    valid=(ds.targets==1).clone()
    valid[list(set(excluded_indices)|{source_idx})]=False
    raw=ds.data[valid].float().reshape(-1,784)/127.5-1
    xc=x.cpu().reshape(-1)
    distances=torch.linalg.vector_norm(raw-xc[None,:],dim=1)
    near=raw[torch.topk(distances,min(n_neighbors,len(distances)),largest=False).indices]-xc[None,:]
    # Neighbor local vectors; not used for scale/threshold construction except
    # via the observable proxy quality on calibration images.
    _,_,vt=torch.linalg.svd(near,full_matrices=False)
    gen=torch.Generator(device='cpu').manual_seed(1911+int(source_idx))
    vecs={'translation':[vx,vy], 'neighbors':[vt[0].to(device),vt[1].to(device)],
          'random':[torch.randn(784,generator=gen).to(device),
                    torch.randn(784,generator=gen).to(device)]}
    return vecs


def tangent_proxy(op, vecs, steps=None):
    """Finite-noise proxy risk in [0,1]. Lower is better.

    Q is an approximate NORMAL operator. It should suppress legitimate
    tangent directions more than matched random directions. This criterion
    does not require the raw Q eigenvalues to be exactly 0 or 1, and uses
    one Q-vector product per direction (much less than Lanczos filtering).
    """
    out={}
    for group,dirs in vecs.items():
        energies=[]
        for vec in dirs:
            v=vec.float().flatten().to(op.device)
            v=v/torch.linalg.vector_norm(v).clamp_min(1e-12)
            qv=op.q_sym(v)
            energies.append(float(torch.sum(qv**2).item()))
        out[group+'_normal_energy']=float(np.mean(energies))
    baseline=out['random_normal_energy']
    # All-near-zero operators are not evidence of useful normal geometry.
    if baseline<=1e-8:
        out['translation_relative_normal_energy']=1.0
        out['neighbors_relative_normal_energy']=1.0
        out['proxy_risk']=1.0
    else:
        out['translation_relative_normal_energy']=float(np.clip(out['translation_normal_energy']/baseline,0,1))
        out['neighbors_relative_normal_energy']=float(np.clip(out['neighbors_normal_energy']/baseline,0,1))
        out['proxy_risk']=float(PROXY_WEIGHT_TRANSLATION*out['translation_relative_normal_energy']
                              +PROXY_WEIGHT_NEIGHBORS*out['neighbors_relative_normal_energy'])
    return out


def denoise_noise_mse(net,x,op,seed,replicates=2):
    """Actual held-out epsilon prediction risk for one known Gaussian draw.
    This is score-*prediction* performance, NOT score Jacobian error.
    """
    values=[]
    for rep in range(replicates):
        gen=torch.Generator(device='cpu').manual_seed(seed+100003*rep)
        noise=torch.randn(x.shape,generator=gen).to(x.device)
        xt=op.sqrt_ab*x + op.scale*noise
        ids=torch.tensor([op.t],device=x.device,dtype=torch.long)
        classes=torch.ones(1,device=x.device,dtype=torch.long)
        with torch.no_grad():
            pred=net(xt,ids,class_labels=classes).sample
        values.append(float(torch.mean((pred-noise)**2).item()))
    return float(np.mean(values))


def evaluate_anchor(ds,idx,anchor_type,net,device,cfg,t_denoise,exp_seed,excluded_indices):
    clean=image_tensor(ds,idx,device)
    anchor=clean if anchor_type=='original' else denoise_once(net,clean,t_denoise)[0]
    ops=sorted([geom.ScoreOperator(net,anchor,t,cfg['fd_eps']) for t in cfg['timesteps']],
               key=lambda x:x.sigma)
    records=[]
    # Exactly the same core features as the existing Colab.
    for op in ops:
        sp=geom.spectral_data(op,cfg['lanczos_steps'],cfg['trace_probes'],seed=1000+idx)
        records.append(dict(anchor=anchor_type,index=idx,timestep=op.t,sigma=op.sigma,**sp))
    gd=geom.geometry_data(ops,probes=cfg['geom_probes'],seed=7020+idx)
    vecs=heldout_directions(clean,ds,idx,device,excluded_indices,n_neighbors=cfg['neighbors'])
    for op,r,g in zip(ops,records,gd):
        r.update(g)
        r.update(tangent_proxy(op,vecs,cfg['filter_steps']))
        # For fair comparisons, always evaluate epsilon risk with a noise draw
        # starting from the original CLEAN image, not the denoised anchor.
        r['eps_prediction_mse']=denoise_noise_mse(net,clean,op,seed=exp_seed+100*idx+op.t)
        if not np.isfinite(np.asarray([r[k] for k in ('proxy_risk','projector_defect','range_violation','eps_prediction_mse')])).all():
            raise RuntimeError(f'Non-finite diagnostics for image {idx}, timestep {op.t}')
    return records


def construct_cutoffs(fit):
    """Find candidate defect/range thresholds using FIT images only.
    No proxy-risk labels, target image, or dimension estimates are consulted.
    """
    # One threshold pair for all noise scales, not digit-1 test tuning per t.
    ds=np.quantile(fit['projector_defect'],QUANTILES)
    rs=np.quantile(fit['range_violation'],QUANTILES)
    return [(float(d),float(r)) for d in ds for r in rs]


def policy(group, cutoff, mse_gates=None):
    """Use the old cross-scale/symmetry/transition screens unchanged.
    Replace ONLY projector-defect and spectral-range thresholds with an
    independently calibrated candidate pair.
    """
    rows=group.sort_values('sigma').to_dict('records')
    dlim,rlim=cutoff
    for r in rows:
        r['valid']=(r['projector_defect']<=dlim and
           r['range_violation']<=rlim and
           r['cross_scale']<=geom.SELECT['cross'] and
           r['antisymmetry']<=geom.SELECT['asym'] and
           r['transition']<=geom.SELECT['transition'] and
           (mse_gates is None or r['eps_prediction_mse']<=mse_gates[int(r['timestep'])]) and
           .5<r['d_eff']<783.5)
    segments=[];segment=[]
    for i,r in enumerate(rows):
        if r['valid']: segment.append(i)
        elif segment: segments.append(segment);segment=[]
    if segment:segments.append(segment)
    segments=[s for s in segments if len(s)>=geom.SELECT['min_run']]
    if not segments: return None
    metric=lambda rr: rr['transition']+rr['projector_defect']+rr['cross_scale']
    preferred=min(segments,key=lambda g:(np.mean([metric(rows[i]) for i in g]),-len(g)))
    return rows[min(preferred,key=lambda i:metric(rows[i]))]


def calibration_rows(frame,thresholds,delta,cap,mse_gates):
    """Uniform risk-UCB across a *finite, FIT-fixed* threshold grid.
    Conditional Hoeffding bound (bounded proxy loss) with Bonferroni union
    over candidate policies. Calibration anchors, not candidate rows, are iid.
    """
    items=[]
    n=frame['index'].nunique()
    K=len(thresholds)
    for j,(dlim,rlim) in enumerate(thresholds):
        results=[]
        for idx,g in frame.groupby('index'):
            chosen=policy(g,(dlim,rlim),mse_gates)
            if chosen is not None:results.append(chosen)
        m=len(results)
        risk=float(np.mean([z['proxy_risk'] for z in results])) if m else np.nan
        ucb=min(1.,risk+math.sqrt(math.log(K/delta)/(2*m))) if m else 1.
        items.append(dict(candidate=j,defect_cutoff=dlim,range_cutoff=rlim,
                   calibrate_count=n,accepted_count=m,coverage=m/n,
                   mean_proxy_risk=risk,proxy_risk_UCB=ucb,
                   meets_preset_risk_target=bool(m>0 and ucb<=cap)))
    return pd.DataFrame(items)


def select_policy(curve):
    feasible=curve[curve['meets_preset_risk_target']]
    if feasible.empty:return None
    return feasible.sort_values(['coverage','proxy_risk_UCB','defect_cutoff'],
                                ascending=[False,True,True]).iloc[0]


def main(args):
    torch.manual_seed(616);np.random.seed(616)
    torch.set_num_threads(min(4,os.cpu_count() or 2))
    if args.target != 2:
        print('Using an explicitly supplied target index; it remains excluded from the calibration split.')
    device='cuda' if args.device=='auto' and torch.cuda.is_available() else ('cpu' if args.device=='auto' else args.device)
    cfg=(dict(timesteps=[5,35,100],lanczos_steps=8,trace_probes=2,geom_probes=2,
              filter_steps=8,fd_eps=.04,neighbors=40) if args.mode=='pilot' else
         dict(timesteps=[5,15,35,65,100],lanczos_steps=16,trace_probes=4,geom_probes=3,
              filter_steps=16,fd_eps=.04,neighbors=60))
    dataset,fit_indices,cal_indices=get_test_images(args.target,args.fit,args.cal)
    anchors=['original','denoised'] if args.anchors=='both' else ['original']
    excluded=set([args.target]+fit_indices+cal_indices)
    net=geom.model_load(device)
    out=Path(args.out);out.mkdir(parents=True,exist_ok=True)
    protocol=dict(target_index=args.target,fit_indices=fit_indices,calibration_indices=cal_indices,
      test_image_used_to_tune=False,checkpoint=geom.MODEL_ID,
      mode=args.mode,anchor_types=anchors,ddpm_timestep_for_denoising=35,
      fit_quantiles=QUANTILES,delta=args.delta,max_proxy_risk=args.max_proxy_risk,
      proxy_definition='0.5*min(1,||Q translation||^2/||Q random||^2)+0.5*min(1,||Q neighbor||^2/||Q random||^2); near-zero Q random energy is assigned risk=1',
      risk_bound='finite-grid uniform Hoeffding UCB over independently sampled calibration anchor images',
      excludes_dimension_ground_truth=True,
      original_fixed_cutoffs={k:geom.SELECT[k] for k in geom.SELECT},
      random_noise_repetitions_for_mse=2,
      independent_neighbor_reference_excludes_target_fit_and_calibration=True)
    (out/'calibration_protocol.json').write_text(json.dumps(protocol,indent=2))
    for anchor in anchors:
        print(f'\n===== CALIBRATING {anchor.upper()} ANCHOR =====',flush=True)
        rows=[]
        for name,indices in [('fit',fit_indices),('calibration',cal_indices)]:
            for n,idx in enumerate(indices):
                print(f'  {name:11s} {n+1:>3d}/{len(indices)}: image {idx}',flush=True)
                data=evaluate_anchor(dataset,idx,anchor,net,device,cfg,35,6151,excluded)
                for d in data:d['split']=name
                rows.extend(data)
        df=pd.DataFrame(rows)
        df.to_csv(out/f'{anchor}_all_calibration_features.csv',index=False)
        fit=df[df.split=='fit'];cal=df[df.split=='calibration']
        thresholds=construct_cutoffs(fit)
        # Model-error gate: noise-dependent 90th percentile of held-out
        # epsilon-prediction MSE on FIT split only. Never fitted on target.
        mse_gates={int(t):float(v) for t,v in fit.groupby('timestep')['eps_prediction_mse'].quantile(.90).items()}
        (out/f'{anchor}_mse_gates.json').write_text(json.dumps(mse_gates,indent=2))
        curve=calibration_rows(cal,thresholds,args.delta,args.max_proxy_risk,mse_gates)
        curve.to_csv(out/f'{anchor}_risk_coverage_calibration.csv',index=False)
        chosen=select_policy(curve)
        target_rows=evaluate_anchor(dataset,args.target,anchor,net,device,cfg,35,6151,excluded)
        target=pd.DataFrame(target_rows)
        target.to_csv(out/f'{anchor}_TARGET_scale_features.csv',index=False)
        if chosen is None:
            result={'anchor':anchor,'status':'ABSTAIN',
                    'reason':'No FIT-defined cutoff passed simultaneous calibrated proxy-risk bound.',
                    'calibrated_proxy_risk_bound':None,
                    'target_index':args.target}
        else:
            dlim=float(chosen['defect_cutoff']);rlim=float(chosen['range_cutoff'])
            picked=policy(target,(dlim,rlim),mse_gates)
            result=dict(anchor=anchor,status='ACCEPT' if picked is not None else 'ABSTAIN',
                     defect_cutoff=dlim,range_cutoff=rlim,
                     noise_scale_specific_mse_gates=mse_gates,
                     calibration_coverage=float(chosen['coverage']),
                     calibration_mean_proxy_risk=float(chosen['mean_proxy_risk']),
                     calibrated_proxy_risk_bound=float(chosen['proxy_risk_UCB']),
                     target_index=args.target,
                     target_timestep=(int(picked['timestep']) if picked else None),
                     target_sigma=(float(picked['sigma']) if picked else None),
                     target_dimension_estimate=(float(picked['d_eff']) if picked else None),
                     target_proxy_risk_POSTHOC=(float(picked['proxy_risk']) if picked else None))
        (out/f'{anchor}_calibrated_target_result.json').write_text(json.dumps(result,indent=2))
        fig,axs=plt.subplots(1,2,figsize=(10,4))
        axs[0].scatter(curve.coverage,curve.mean_proxy_risk,alpha=.8,label='Empirical proxy risk')
        axs[0].scatter(curve.coverage,curve.proxy_risk_UCB,alpha=.8,label='Uniform upper bound')
        axs[0].axhline(args.max_proxy_risk,ls='--',color='black',label='Precommitted risk target')
        axs[0].set(xlabel='Calibration acceptance coverage',ylabel='Proxy tangent risk',ylim=(-.05,1.05))
        axs[0].legend(fontsize=7);axs[0].grid(alpha=.25)
        axs[1].scatter(curve.defect_cutoff,curve.range_cutoff,c=curve.coverage,
                       cmap='viridis',s=75,vmin=0,vmax=1)
        axs[1].set(xlabel='Calibrated projector defect cutoff',ylabel='Calibrated range cutoff')
        axs[1].grid(alpha=.25)
        fig.suptitle(anchor+': fit/calibration are disjoint; target unseen during fitting')
        fig.tight_layout();fig.savefig(out/f'{anchor}_calibration_figure.png',dpi=150);plt.close(fig)
        print('\n'+json.dumps(result,indent=2),flush=True)
        print('Calibration frontier (highest coverage first):\n'+curve.sort_values('coverage',ascending=False).head(6).to_string(index=False),flush=True)
    readme='''# Calibrated finite-noise MNIST geometry — protocol\n\n'''
    readme+='- Uses **distinct digit-1 test images** for threshold fit, risk calibration, and target image.\n'
    readme+='- Calibrates **observed Q-normal suppression proxy risk**: real translation and independent neighbor directions must be more strongly suppressed than matched random directions. NOT true intrinsic dimension or correctness.\n'
    readme+='- Uses the same pretrained conditional DDPM and same base spectral filter.\n'
    readme+='- Fitting quantiles determine candidate projector-defect and range thresholds. No target data is used.\n'
    readme+='- A held-out risk-calibration split selects among these fixed candidates using a uniform Hoeffding upper bound.\n'
    readme+='- A separately FIT-calibrated noise-level-specific gate measures average epsilon prediction error over two known Gaussian noise draws; this is not Jacobian error.\n'
    readme+='- The confidence statement requires independent exchangeable calibration anchors, a bounded preregistered proxy loss, and model/criteria fixed before calibration.\n'
    readme+='- A pilot with few images may correctly return ABSTAIN because finite-sample bounds are loose.\n'
    readme+='- Original and denoised anchors have separate calibration runs, so no distribution-shift guarantee is transferred between them.\n'
    readme+='- Without an actual known MNIST manifold dimension, DO NOT claim calibrated dimension MAE or true geometry certification.\n'
    readme+='- Accuracy improvement over official LHSD remains untested.\n'
    (out/'CALIBRATION_INTERPRETATION.md').write_text(readme)
    print('\nSaved results under',out,flush=True)


if __name__=='__main__':
    p=argparse.ArgumentParser()
    p.add_argument('--mode',choices=['pilot','expanded'],default='pilot')
    p.add_argument('--target',type=int,default=2)
    p.add_argument('--fit',type=int,default=6)
    p.add_argument('--cal',type=int,default=12)
    p.add_argument('--anchors',choices=['original','both'],default='original')
    p.add_argument('--max-proxy-risk',type=float,default=MAX_PROXY_RISK)
    p.add_argument('--delta',type=float,default=CONFIDENCE_DELTA)
    p.add_argument('--device',choices=['auto','cpu','cuda'],default='auto')
    p.add_argument('--out',default='/content/mnist_reliability_calibration')
    args=p.parse_args()
    if not 0<args.max_proxy_risk<=1 or not 0<args.delta<1:raise ValueError('Risk target and delta must be in (0,1).')
    if args.fit<4 or args.cal<4:raise ValueError('Use at least four independent anchors in each split.')
    main(args)


In [ ]:
%%writefile /content/MNIST_Scale_Selection_Repair.py
#!/usr/bin/env python3
"""Correct mismatch between geometry-proxy risk and scale selection.

No classifier/surrogate used. Reuses exact frozen score diagnostics from existing pilot.
- Reconstructs the previous 40+500 image split (seed=7604) so NEW images are disjoint.
- Fits timestep-wise expected proxy risk on FIT-only images.
- Compares legacy selection against FIT-ranked selection at the SAME FIT-defined
  16 projector/range cutoff pairs and the SAME eligibility constraints.
- Simultaneous bounded-loss Hoeffding UCB on untouched calibration images.
- Writes each completed image immediately; reruns resume without recomputation.
- Does NOT claim true intrinsic-dimension/tangent-projector certification.
"""
import argparse
import json
import math
import os
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import MNIST_Digit1_Final_Geometry as geom
import MNIST_Reliability_Calibration_Addon as base


def reconstruct_disjoint_split(target, old_fit, old_cal, n_fit, n_cal):
    from torchvision.datasets import MNIST
    ds = MNIST('/content/mnist_geometry', train=False, download=True)
    if int(ds.targets[target]) != 1:
        raise ValueError('Target must be a digit 1 in MNIST test set')
    all_ones = [int(i) for i in torch.where(ds.targets == 1)[0] if int(i) != target]
    old_rng = np.random.default_rng(7604)
    old_rng.shuffle(all_ones)
    previously_seen = set(all_ones[:old_fit + old_cal])
    fresh = [int(i) for i in all_ones if int(i) not in previously_seen]
    rng = np.random.default_rng(20261102)
    rng.shuffle(fresh)
    if len(fresh) < n_fit + n_cal:
        raise ValueError(f'Only {len(fresh)} unseen digit-1 test images remain; requested {n_fit+n_cal}.')
    return ds, fresh[:n_fit], fresh[n_fit:n_fit+n_cal], previously_seen, len(fresh)


def reconstruct_fit_ranking(fit):
    # Each score is fitted on FIT images only, never on evaluation images.
    # Rank by empirical proxy risk. This is a proxy-driven selector, NOT a
    # dimension-accuracy selector; separate tangent validation is required.
    means = fit.groupby('timestep')['proxy_risk'].mean()
    if not means.index.is_unique or means.isna().any():
        raise ValueError('Missing or invalid fitted timestep proxy risk')
    return {int(t): float(r) for t, r in means.items()}


def proxy_ranked_policy(group, cutoff, mse_gates, fit_risks):
    rows = group.sort_values('sigma').to_dict('records')
    dlim, rlim = cutoff
    for r in rows:
        t = int(r['timestep'])
        r['valid'] = bool(
            np.isfinite(r['proxy_risk']) and np.isfinite(r['d_eff']) and
            r['projector_defect'] <= dlim and
            r['range_violation'] <= rlim and
            r['cross_scale'] <= geom.SELECT['cross'] and
            r['antisymmetry'] <= geom.SELECT['asym'] and
            r['transition'] <= geom.SELECT['transition'] and
            r['eps_prediction_mse'] <= mse_gates[t] and
            0.5 < r['d_eff'] < 783.5
        )
    # Keep *identical* at-least-two-consecutive-scales eligibility rule.
    qualifying = []
    run = []
    for i,r in enumerate(rows):
        if r['valid']:
            run.append(i)
        elif run:
            if len(run) >= geom.SELECT['min_run']:
                qualifying.extend(run)
            run = []
    if len(run) >= geom.SELECT['min_run']:
        qualifying.extend(run)
    if not qualifying:
        return None
    # No held-out image's proxy risk is used for deciding which scale to pick.
    # Ties resolved with an internal score that doesn't inspect proxy risk.
    metric = lambda r: r['transition'] + r['projector_defect'] + r['cross_scale']
    chosen = min(qualifying,
        key=lambda i: (fit_risks[int(rows[i]['timestep'])], metric(rows[i])))
    return rows[chosen]


def risk_coverage_table(cal, thresholds, mse_gates, fit_risks, delta, cap):
    # Exactly 16 candidates per method. Uniform multiplicity correction 32.
    candidates = [(kind,i,cutoff) for kind in ('legacy','proxy_ranked')
                  for i,cutoff in enumerate(thresholds)]
    K = len(candidates)
    n = int(cal['index'].nunique())
    by_image = list(cal.groupby('index'))
    results = []
    for kind,i,cutoff in candidates:
        selected = []
        for idx,group in by_image:
            if kind == 'legacy':
                row = base.policy(group, cutoff, mse_gates)
            else:
                row = proxy_ranked_policy(group, cutoff, mse_gates, fit_risks)
            if row is not None:
                selected.append(row)
        m = len(selected)
        mean = float(np.mean([r['proxy_risk'] for r in selected])) if m else np.nan
        se = math.sqrt(math.log(K/delta)/(2*m)) if m else np.inf
        ucb = float(min(1,mean+se)) if m else 1.
        results.append(dict(method=kind, candidate=i, defect_cutoff=cutoff[0],
          range_cutoff=cutoff[1], n_calibration=n, accepted_count=m,
          coverage=m/n, mean_proxy_risk=mean, proxy_risk_UCB=ucb,
          meets_preset_risk_target=bool(m>0 and ucb<=cap),
          mean_selected_timestep=(float(np.mean([r['timestep'] for r in selected])) if m else np.nan)))
    return pd.DataFrame(results)


def save_completed_image(ds, idx, split, net, device, cfg, excluded, out):
    # One atomic CSV per image. If Colab disconnects, completed samples survive.
    save = out / 'per_image' / f'{split}_index_{idx}.csv'
    save.parent.mkdir(parents=True, exist_ok=True)
    if save.exists():
        rows = pd.read_csv(save)
        if set(rows['timestep']) == set(cfg['timesteps']) and len(rows)==len(cfg['timesteps']):
            return rows
    values = base.evaluate_anchor(ds, idx, 'original', net, device, cfg,
                                  t_denoise=35, exp_seed=6151,
                                  excluded_indices=excluded)
    for row in values: row['split'] = split
    rows = pd.DataFrame(values)
    temporary = save.with_suffix('.tmp.csv')
    rows.to_csv(temporary,index=False)
    os.replace(temporary, save)
    return rows


def main(args):
    torch.manual_seed(616)
    np.random.seed(616)
    torch.set_num_threads(min(4,os.cpu_count() or 2))
    ds, fit_ids, cal_ids, old_ids, remaining = reconstruct_disjoint_split(
        args.target, args.previous_fit, args.previous_cal, args.fit, args.cal)
    out = Path(args.out)
    out.mkdir(parents=True,exist_ok=True)
    cfg = dict(timesteps=[5,35,100],lanczos_steps=8,trace_probes=2,
               geom_probes=2,filter_steps=8,fd_eps=.04,neighbors=40)
    protocol = dict(previous_fit=args.previous_fit, previous_cal=args.previous_cal,
       old_seed=7604, new_seed=20261102, target=args.target, remaining_after_old_run=remaining,
       new_fit_indices=fit_ids, new_calibration_indices=cal_ids,
       prev_ids_excluded=True, target_excluded=True,
       purpose='test a new FIT-ranked geometry-proxy scale-selector; not to validate true dimension',
       candidate_count=2*len(base.QUANTILES)**2, delta=args.delta,
       risk_cap=args.max_proxy_risk,
       caveat='New policy is post-hoc motivated by old 500-image exploratory result; these new images have never been evaluated in the old experiment')
    (out/'REPAIR_PROTOCOL.json').write_text(json.dumps(protocol,indent=2))
    device = 'cuda' if args.device=='auto' and torch.cuda.is_available() else (
        'cpu' if args.device=='auto' else args.device)
    net = None  # Lazy: a resumed, fully cached run does not need to reload the model.
    # Neighbors exclude every selected sample and the old split. This leaves
    # a smaller independent reference pool but prevents identity overlap.
    excluded = set([args.target]) | old_ids | set(fit_ids) | set(cal_ids)
    rows=[]
    for split,ids in [('fit',fit_ids),('calibration',cal_ids)]:
        for j,idx in enumerate(ids):
            print(f'{split:11s} {j+1:>3d}/{len(ids)} | digit-1 index {idx}',flush=True)
            saved=out/'per_image'/f'{split}_index_{idx}.csv'
            if not saved.exists() and net is None:
                net=geom.model_load(device)
            rows.append(save_completed_image(ds,idx,split,net,device,cfg,excluded,out))
    frame = pd.concat(rows,ignore_index=True)
    frame.to_csv(out/'ALL_FINISHED_FEATURES.csv',index=False)
    fit = frame[frame.split=='fit'].copy()
    cal = frame[frame.split=='calibration'].copy()
    fit_risk = reconstruct_fit_ranking(fit)
    cutoffs = base.construct_cutoffs(fit)
    mse_gates = {int(t):float(v) for t,v in
       fit.groupby('timestep')['eps_prediction_mse'].quantile(.90).items()}
    (out/'FIT_SCALE_RISK.json').write_text(json.dumps(fit_risk,indent=2))
    (out/'FIT_MSE_GATES.json').write_text(json.dumps(mse_gates,indent=2))
    curve = risk_coverage_table(cal,cutoffs,mse_gates,fit_risk,
                                 args.delta,args.max_proxy_risk)
    curve.to_csv(out/'REPAIRED_vs_LEGACY_RISK_COVERAGE.csv',index=False)
    # Contrast always at equal coverage where possible; otherwise show all.
    outcomes = {}
    for kind,g in curve.groupby('method'):
        feasible = g[g.meets_preset_risk_target]
        if feasible.empty:
            print(f'\n{kind}: ABSTAIN (no feasible policy)')
            outcomes[kind]={'status':'ABSTAIN'}
        else:
            chosen = feasible.sort_values(['coverage','proxy_risk_UCB'],ascending=[False,True]).iloc[0]
            print(f'\n{kind}: {chosen.to_dict()}')
            cutoff=(float(chosen['defect_cutoff']),float(chosen['range_cutoff']))
            selections=[]
            for idx, group in cal.groupby('index'):
                rr=(base.policy(group, cutoff, mse_gates) if kind=='legacy' else
                    proxy_ranked_policy(group,cutoff,mse_gates,fit_risk))
                if rr is not None:
                    selections.append(dict(index=int(idx),timestep=int(rr['timestep']),
                        d_eff=float(rr['d_eff']),proxy_risk=float(rr['proxy_risk'])))
            pd.DataFrame(selections).to_csv(out/f'{kind}_SELECTED_IMAGES.csv',index=False)
            outcomes[kind]={'status':'ACCEPT','coverage':float(chosen['coverage']),
                'mean_proxy_risk':float(chosen['mean_proxy_risk']),
                'proxy_risk_UCB':float(chosen['proxy_risk_UCB']),
                'timestep_counts':{str(k):int(v) for k,v in
                    pd.DataFrame(selections)['timestep'].value_counts().items()},
                'median_dimension':float(pd.DataFrame(selections)['d_eff'].median())}
    (out/'REPAIR_CONCLUSION.json').write_text(json.dumps(outcomes,indent=2))
    print('\nFITTED TIMESTEP PROXY RISKS:',fit_risk)
    print('\nCALIBRATION RAW PROXY MEAN BY SCALE:')
    print(cal.groupby('timestep')['proxy_risk'].agg(['mean','median','std']).to_string())
    print(f'\nOutput persisted at {out}. Future runs RESUME per completed image.')


if __name__=='__main__':
    p=argparse.ArgumentParser()
    p.add_argument('--target',type=int,default=2)
    p.add_argument('--previous-fit',type=int,default=40)
    p.add_argument('--previous-cal',type=int,default=500)
    p.add_argument('--fit',type=int,default=20)
    p.add_argument('--cal',type=int,default=100)
    p.add_argument('--max-proxy-risk',type=float,default=.65)
    p.add_argument('--delta',type=float,default=.10)
    p.add_argument('--device',choices=['auto','cpu','cuda'],default='auto')
    p.add_argument('--out',default='/content/drive/MyDrive/MNIST_scale_repair')
    a=p.parse_args()
    if a.fit<4 or a.cal<4: raise ValueError('Need at least four fit/cal images')
    if not 0<a.delta<1 or not 0<a.max_proxy_risk<=1: raise ValueError('delta and cap must be in (0,1)')
    main(a)


## B. Operator self-tests (no checkpoint download)
This checks the old numerical Jacobian, adjoint and Lanczos spectral filter before any expensive inference.

In [ ]:
!python /content/MNIST_Digit1_Final_Geometry.py --self-test

## C. Mount Drive and locate the **original**, cached FIT/CAL run
The notebook does **not** treat existing calibration points as new test points. It reconstructs the same policy from immutable saved features and verifies the saved results. Set `RUN_REPAIR_IF_MISSING=True` **only** if your original files are absent (a much more expensive restart).

In [ ]:
from pathlib import Path
from google.colab import drive
import os, sys, json, math, hashlib, importlib, subprocess
import numpy as np, pandas as pd, torch
from IPython.display import display
from torchvision.datasets import MNIST

drive.mount('/content/drive')
sys.path.insert(0, '/content')
import MNIST_Digit1_Final_Geometry as geom
import MNIST_Reliability_Calibration_Addon as base
import MNIST_Scale_Selection_Repair as repair
importlib.reload(geom); importlib.reload(base); importlib.reload(repair)

REPAIR_DIR = Path('/content/drive/MyDrive/MNIST_scale_repair')
OUTPUT_DIR = Path('/content/drive/MyDrive/MNIST_calibrated_task2')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_REPAIR_IF_MISSING = False    # use saved 20 FIT + 100 CAL points, no 45-min rerun
N_NEW_TEST = 3                 # safe pilot; later increase to 5-20
N_RANDOM_PROBES = 4
LANCZOS_STEPS = 8
SHIFT_PIXELS = 0.10
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE != 'cuda':
    print('WARNING: CPU mode may be very slow; choose T4 GPU in Colab.')
torch.set_num_threads(min(4, os.cpu_count() or 2))

needed = ['REPAIR_PROTOCOL.json','ALL_FINISHED_FEATURES.csv',
          'REPAIRED_vs_LEGACY_RISK_COVERAGE.csv','FIT_SCALE_RISK.json',
          'FIT_MSE_GATES.json','REPAIR_CONCLUSION.json']
missing = [n for n in needed if not (REPAIR_DIR/n).is_file()]
if missing and RUN_REPAIR_IF_MISSING:
    print('Repair artifacts missing:', missing, '| running original checkpoint-resumable repair...')
    subprocess.run([sys.executable,'/content/MNIST_Scale_Selection_Repair.py',
        '--fit','20','--cal','100','--previous-fit','40','--previous-cal','500',
        '--target','2','--device','auto','--out',str(REPAIR_DIR)],check=True)
    missing = [n for n in needed if not (REPAIR_DIR/n).is_file()]
if missing:
    raise FileNotFoundError(
        'The earlier completed calibration run was NOT found in '+str(REPAIR_DIR)+
        '\nMissing: '+', '.join(missing)+
        '\nCheck the Google Drive folder in the Colab Files sidebar or update REPAIR_DIR. '
        'Only if you truly lost the run, change RUN_REPAIR_IF_MISSING=True and rerun this cell.')
print('Using existing repair artifacts:', REPAIR_DIR)
print('New test output:', OUTPUT_DIR)

## D. Reconstruct and **freeze** the previously calibrated policy
Preserves: 3 scales `5/35/100`, FIT-only risk ranking, FIT quantile grid (16 pairs), FIT-only MSE gates, original consecutive-scale eligibility, simultaneous Hoeffding UCB across 32 candidate policies, `delta=0.10`, risk cap `0.65`. **Never refit on new test images.**

In [ ]:
repair_protocol = json.loads((REPAIR_DIR/'REPAIR_PROTOCOL.json').read_text())
frame = pd.read_csv(REPAIR_DIR/'ALL_FINISHED_FEATURES.csv')
saved_curve = pd.read_csv(REPAIR_DIR/'REPAIRED_vs_LEGACY_RISK_COVERAGE.csv')
saved_conclusion = json.loads((REPAIR_DIR/'REPAIR_CONCLUSION.json').read_text())
fit_ids = [int(v) for v in repair_protocol['new_fit_indices']]
cal_ids = [int(v) for v in repair_protocol['new_calibration_indices']]
assert len(fit_ids)==20 and len(cal_ids)==100, 'Saved run is not the previously reported 20/100 protocol.'
assert set(fit_ids).isdisjoint(cal_ids)
assert int(repair_protocol['target'])==2
assert set(frame['timestep'].astype(int))=={5,35,100}, 'Do not reuse a policy fitted with a different scale grid.'
assert set(frame[frame['split']=='fit']['index'].astype(int))==set(fit_ids)
assert set(frame[frame['split']=='calibration']['index'].astype(int))==set(cal_ids)
assert frame.groupby(['split','index']).size().eq(3).all(), 'Each calibration anchor must have exactly 3 scales.'
assert 'proxy_risk' in frame and frame['proxy_risk'].between(0,1).all()
assert int(repair_protocol['candidate_count'])==32
DELTA=float(repair_protocol['delta'])
RISK_CAP=float(repair_protocol['risk_cap'])
assert abs(DELTA-0.10)<1e-12 and abs(RISK_CAP-0.65)<1e-12

fit_frame=frame[frame.split=='fit'].copy()
cal_frame=frame[frame.split=='calibration'].copy()
fit_risks=repair.reconstruct_fit_ranking(fit_frame)
cutoffs=base.construct_cutoffs(fit_frame)
mse_gates={int(t):float(v) for t,v in
    fit_frame.groupby('timestep')['eps_prediction_mse'].quantile(.90).items()}
with open(REPAIR_DIR/'FIT_SCALE_RISK.json') as f: saved_fit=json.load(f)
with open(REPAIR_DIR/'FIT_MSE_GATES.json') as f: saved_mse=json.load(f)
for k,v in fit_risks.items():
    assert np.isclose(v,float(saved_fit[str(k)]),atol=1e-8)
for k,v in mse_gates.items():
    assert np.isclose(v,float(saved_mse[str(k)]),atol=1e-8)
curve=repair.risk_coverage_table(cal_frame,cutoffs,mse_gates,fit_risks,DELTA,RISK_CAP)
cols=['coverage','mean_proxy_risk','proxy_risk_UCB','defect_cutoff','range_cutoff']
assert len(curve)==32 and len(saved_curve)==32
assert (curve[['method','candidate']].values==saved_curve[['method','candidate']].values).all()
for col in cols:
    assert np.allclose(curve[col],saved_curve[col],equal_nan=True,rtol=1e-5,atol=1e-8),f'Mismatch in {col}; refusing to change policy silently.'

eligible=curve[(curve.method=='proxy_ranked') & (curve.meets_preset_risk_target)].copy()
if eligible.empty:
    SELECTED_POLICY=None
    print('CALIBRATED SELECTOR ABSTAINS GLOBALLY: no previously accepted policy found.')
else:
    SELECTED_POLICY=eligible.sort_values(['coverage','proxy_risk_UCB'],ascending=[False,True]).iloc[0]
    assert saved_conclusion['proxy_ranked']['status']=='ACCEPT'
    assert np.isclose(SELECTED_POLICY['proxy_risk_UCB'], saved_conclusion['proxy_ranked']['proxy_risk_UCB'])
    print('FROZEN FIT-ranked policy:')
    display(SELECTED_POLICY.to_frame().T)
    print('Frozen FIT scale risk:',fit_risks,'| previous calibration coverage:',float(SELECTED_POLICY['coverage']))

# Save the exact policy and provenance separately from immutable repair data.
sha=hashlib.sha256((REPAIR_DIR/'ALL_FINISHED_FEATURES.csv').read_bytes()).hexdigest()
frozen = dict(source_dir=str(REPAIR_DIR),data_sha256=sha,model_id=geom.MODEL_ID,
    calibration_type='bounded MNIST handwriting-direction PROXY risk (NOT true tangent error)',
    original_previous_fit=40,original_previous_cal=500,
    fit_indices=fit_ids,cal_indices=cal_ids,delta=DELTA,risk_cap=RISK_CAP,
    fit_scale_risks=fit_risks,mse_gates=mse_gates,
    selection_status='ACCEPT' if SELECTED_POLICY is not None else 'ABSTAIN',
    calibrated_policy=None if SELECTED_POLICY is None else dict(
        defect_cutoff=float(SELECTED_POLICY['defect_cutoff']),
        range_cutoff=float(SELECTED_POLICY['range_cutoff']),
        proxy_risk_UCB=float(SELECTED_POLICY['proxy_risk_UCB']),
        observed_cal_coverage=float(SELECTED_POLICY['coverage']),
        accepted_cal_count=int(SELECTED_POLICY['accepted_count'])),
    immutable_thresholds=geom.SELECT,
    no_further_tuning_on_test=True)
(OUTPUT_DIR/'FROZEN_POLICY.json').write_text(json.dumps(frozen,indent=2))

## E. New disjoint MNIST test images — no calibration leakage
The selection rule is frozen above. Test images are drawn from digit-1 MNIST test indices disjoint from the old 40+500 exploratory split and the new 20+100 FIT/CAL split. No classifier output is used.

In [ ]:
ds, expected_fit, expected_cal, old_ids, _ = repair.reconstruct_disjoint_split(2,40,500,20,100)
assert list(expected_fit)==fit_ids and list(expected_cal)==cal_ids, 'Inconsistent reconstructed indices.'
excluded_old=set(old_ids)|set(fit_ids)|set(cal_ids)|{2}
remaining=np.array([int(i) for i in torch.where(ds.targets==1)[0].tolist()
                    if int(i) not in excluded_old],dtype=np.int64)
rng=np.random.default_rng(20261103)
new_indices=rng.permutation(remaining)[:N_NEW_TEST].tolist()
assert len(new_indices)==N_NEW_TEST
assert set(new_indices).isdisjoint(excluded_old)
print('Unseen, withheld digit-1 indices:',new_indices)
(OUTPUT_DIR/'TEST_SPLIT.json').write_text(json.dumps(dict(
    indices=new_indices,selection_seed=20261103,
    excluded_previous_540=True,excluded_fit_and_cal_120=True,
    label_source='official MNIST labels, no classifier inference'),indent=2))

## F. Compute new image diagnostics, applying the **frozen** policy
Preserves the same original `evaluate_anchor` function and per-image CSV schema. Each new image's metrics are saved immediately to Drive, allowing resumption after a Colab disconnect. **The new proxy risk is measured for retrospective evaluation; the policy's timestep ranking uses only FIT risks**. A unit check below verifies that altering the numerical held-out proxy values does not change the selected timestep.

In [ ]:
new_eval_dir = OUTPUT_DIR/'per_image'
new_eval_dir.mkdir(parents=True,exist_ok=True)
excluded_references=excluded_old|set(new_indices)
CFG=dict(timesteps=[5,35,100],lanczos_steps=8,trace_probes=2,geom_probes=2,
         filter_steps=8,fd_eps=.04,neighbors=40)
net=None
all_new=[]
for j,idx in enumerate(new_indices):
    file=new_eval_dir/f'test_index_{idx}.csv'
    if file.is_file():
        rows=pd.read_csv(file)
        assert len(rows)==3 and set(rows.timestep.astype(int))=={5,35,100}
        print('Loaded cached',file.name)
    else:
        if net is None: net=geom.model_load(DEVICE)
        vals=base.evaluate_anchor(ds,idx,'original',net,DEVICE,CFG,
             t_denoise=35,exp_seed=6151,excluded_indices=excluded_references)
        rows=pd.DataFrame(vals).assign(split='unseen_test')
        tmp=file.with_suffix('.tmp.csv')
        rows.to_csv(tmp,index=False); os.replace(tmp,file)
        print(f'New diagnostics {j+1}/{len(new_indices)} finished:',file.name,flush=True)
    all_new.append(rows)
heldout=pd.concat(all_new,ignore_index=True)
heldout.to_csv(OUTPUT_DIR/'UNSEEN_TEST_SCALE_DIAGNOSTICS.csv',index=False)
cutoff=None if SELECTED_POLICY is None else (
    float(SELECTED_POLICY['defect_cutoff']),float(SELECTED_POLICY['range_cutoff']))
selected_rows=[]
for idx,g in heldout.groupby('index'):
    if cutoff is None:
        choice=None
    else:
        choice=repair.proxy_ranked_policy(g,cutoff,mse_gates,fit_risks)
        # Invariance audit: selected timestep must be independent of the held-out
        # proxy-risk values (except the old policy's required finite-value check).
        altered=g.copy()
        altered['proxy_risk']=0.42
        check=repair.proxy_ranked_policy(altered,cutoff,mse_gates,fit_risks)
        assert (choice is None)==(check is None)
        if choice is not None:
            assert int(choice['timestep'])==int(check['timestep'])
    selected_rows.append(dict(index=int(idx),status='ABSTAIN' if choice is None else 'ACCEPT',
         selected_timestep=np.nan if choice is None else int(choice['timestep']),
         selected_proxy_risk=np.nan if choice is None else float(choice['proxy_risk']),
         selected_dimension=np.nan if choice is None else float(choice['d_eff'])))
selection_df=pd.DataFrame(selected_rows)
selection_df.to_csv(OUTPUT_DIR/'UNSEEN_POLICY_DECISIONS.csv',index=False)
display(selection_df)
print('Accepted:',int((selection_df.status=='ACCEPT').sum()),'/',len(selection_df))

## G. Task 2 — save pixel-space candidate tangent direction banks
For every new digit `1`, compute the **same** matrix-free score filter at fixed `t=5`, fixed `t=100` and the calibrated selected scale if accepted. Each bank is generated using **random seeds**, not classifier gradients or neighbor-PCA basis vectors. Save numerical stability and *post-selection* translation/neighbor direction retention. Do not claim exact manifold tangents from these diagnostic scores.

**Note:** If the calibrated policy abstains, we still save `t=5`/`t=100` **exploratory baselines**, but no output is labeled calibrated/accepted.

In [ ]:
import matplotlib.pyplot as plt
import torch.nn.functional as F
if net is None: net=geom.model_load(DEVICE)
NET_ID=geom.MODEL_ID
BANK_DIR=OUTPUT_DIR/'direction_banks'
BANK_DIR.mkdir(parents=True,exist_ok=True)
records=[]; references=[]

for idx in new_indices:
    x=base.image_tensor(ds,idx,DEVICE)
    chosen=selection_df[selection_df['index']==idx].iloc[0]
    calibrated_t=int(chosen['selected_timestep']) if chosen['status']=='ACCEPT' else None
    all_t=sorted(set([5,100]+([calibrated_t] if calibrated_t is not None else [])))
    gen=torch.Generator(device='cpu').manual_seed(20261103+idx)
    raw=F.normalize(torch.randn(N_RANDOM_PROBES,784,generator=gen),dim=1).to(DEVICE)
    xshift=geom.shifted_image(x,SHIFT_PIXELS,0)
    span_by_t={}
    # References are computed only for independent assessment, not bank creation or selection.
    refs=base.heldout_directions(x,ds,idx,DEVICE,excluded_references,n_neighbors=40)

    for t in all_t:
        dest=BANK_DIR/f'digit1_index{idx}_t{t}_directions.pt'
        if dest.is_file():
            packet=torch.load(dest,map_location='cpu',weights_only=True)
            assert packet['model_id']==NET_ID and packet['index']==idx and packet['timestep']==t
            bank=packet['direction_bank'];basis=packet['orthonormal_span']
            strength=packet['filter_strengths']; stable=float(packet['nearby_direction_cosine'])
            print('Reused',dest.name)
        else:
            op=geom.ScoreOperator(net,x,t,fd_eps=0.04)
            filtered=torch.stack([geom.filter_apply(op,v,LANCZOS_STEPS).detach() for v in raw])
            strength=torch.linalg.vector_norm(filtered,dim=1)
            good=strength>1e-6
            bank=F.normalize(filtered[good],dim=1).cpu() if bool(good.any()) else torch.empty(0,784)
            if bank.numel():
                _,sing,vh=torch.linalg.svd(bank,full_matrices=False)
                basis=vh[sing>0.15*sing[0]].contiguous()
            else: basis=torch.empty(0,784)
            nearop=geom.ScoreOperator(net,xshift,t,fd_eps=0.04)
            near=geom.filter_apply(nearop,raw[0],LANCZOS_STEPS).detach()
            if float(strength[0])<=1e-8 or float(near.norm())<=1e-8:
                stable=float('nan')
            else:
                stable=float(F.cosine_similarity(filtered[0][None],near[None]).item())
            packet=dict(index=int(idx),timestep=int(t),sigma_ve=float(op.sigma),
               original_image=x.detach().cpu(),input_random_probes=raw.detach().cpu(),
               direction_bank=bank,orthonormal_span=basis,
               filter_strengths=strength.detach().cpu(),
               nearby_direction_cosine=stable,
               model_id=NET_ID,steps=LANCZOS_STEPS,
               policy_status=str(chosen['status']),
               is_calibrated_selection=bool(calibrated_t is not None and t==calibrated_t),
               calibration_caveat='Calibrated proxy-risk selection, not calibrated true tangent error.',
               note='Heuristic score-filtered pixel-space directions; not certified manifold tangents.')
            torch.save(packet,dest)
            strength=packet['filter_strengths']
            print('Saved',dest.name,flush=True)
        span_by_t[t]=basis
        records.append(dict(index=int(idx),timestep=int(t),
             calibrated_selected=bool(calibrated_t is not None and t==calibrated_t),
             accepted_policy=chosen['status']=='ACCEPT',
             basis_rank=int(basis.shape[0]),usable_bank_size=int(bank.shape[0]),
             mean_strength=float(strength.float().mean()),
             nearby_direction_cosine=float(stable)))
        # Independent reference checks, not input to the bank or timestep ranking.
        op=geom.ScoreOperator(net,x,t,fd_eps=0.04)
        for typ,vecs in refs.items():
            for j,v in enumerate(vecs):
                v=v.flatten().float().to(DEVICE)
                v=v/v.norm().clamp_min(1e-12)
                fv=geom.filter_apply(op,v,LANCZOS_STEPS).detach()
                references.append(dict(index=int(idx),timestep=int(t),group=typ,
                    direction_number=j+1,squared_filter_retention=float(fv.square().sum()),
                    input_output_cosine=float(torch.dot(v,fv)/fv.norm().clamp_min(1e-12))))
    # Principal-angle cosine diagnostic across the two fixed-scale candidate spans.
    b5,b100=span_by_t.get(5),span_by_t.get(100)
    if b5 is not None and b100 is not None and len(b5) and len(b100):
        overlap=torch.linalg.svdvals(b5@b100.T)
        cross=float(overlap.mean())
    else: cross=float('nan')
    for row in records:
        if row['index']==idx:row['t5_vs_t100_mean_principal_cosine']=cross

banks=pd.DataFrame(records)
refchecks=pd.DataFrame(references)
banks.to_csv(OUTPUT_DIR/'DIRECTION_DIAGNOSTICS.csv',index=False)
refchecks.to_csv(OUTPUT_DIR/'INDEPENDENT_DIRECTION_CHECKS.csv',index=False)
display(banks)
print('Reference directional retention:')
display(refchecks.groupby(['timestep','group']).squared_filter_retention.agg(['mean','count']).round(4))

## H. Real MNIST evaluation: **proxy-risk UCB**, stability, and observed held-out risk
This is an untouched-test descriptive evaluation of a policy chosen on FIT/CAL. The previous UCB concerns proxy risk among accepted calibration anchors under the independence/exchangeability/bounded-loss assumptions. It is **not a per-image true-error bound**, nor a certified bound under arbitrary image distribution shift.

In [ ]:
accepted=selection_df[selection_df.status=='ACCEPT']
hold_summary=dict(number_test=len(selection_df),accepted=len(accepted),
  empirical_acceptance_coverage=float(len(accepted)/len(selection_df)),
  mean_observed_heldout_proxy_risk=(float(accepted.selected_proxy_risk.mean()) if len(accepted) else None),
  frozen_calibration_proxy_UCB=(float(SELECTED_POLICY['proxy_risk_UCB']) if SELECTED_POLICY is not None else None),
  frozen_proxy_risk_target=float(RISK_CAP),
  heldout_intrinsic_dimension_ground_truth='UNKNOWN',
  heldout_tangent_projector_ground_truth='UNKNOWN',
  true_error_calibration_status='NOT_IDENTIFIABLE_WITHOUT_VALIDATED_TANGENT_GROUND_TRUTH')
(OUTPUT_DIR/'UNSEEN_PROXY_RISK_EVALUATION.json').write_text(json.dumps(hold_summary,indent=2))
print(json.dumps(hold_summary,indent=2))

fig,ax=plt.subplots(1,2,figsize=(11,4))
counts=selection_df.groupby(['status','selected_timestep'],dropna=False).size()
ax[0].bar(range(len(counts)),counts.values)
ax[0].set_xticks(range(len(counts)),[f'{a}, t={b:g}' if np.isfinite(b) else f'{a}' for a,b in counts.index],rotation=30)
ax[0].set_ylabel('Unseen digit-1 images');ax[0].set_title('Frozen selected scales / abstention')
if len(accepted):
    ax[1].scatter(accepted['index'],accepted.selected_proxy_risk,label='Measured held-out proxy risk')
if SELECTED_POLICY is not None:
    ax[1].axhline(float(SELECTED_POLICY['proxy_risk_UCB']),ls='--',label='Original calibration proxy UCB')
ax[1].axhline(RISK_CAP,ls=':',label='Preset proxy-risk target')
ax[1].set(xlabel='MNIST test index',ylabel='Bounded proxy risk',ylim=(0,1));ax[1].legend(fontsize=8)
fig.tight_layout();fig.savefig(OUTPUT_DIR/'FROZEN_POLICY_HELDOUT_EVALUATION.png',dpi=150)
plt.show()

## I. True tangent-error calibration audit — **do not fabricate ground truth**

The original synthetic conformal predictor could calculate `true_tangent_error` because synthetic points had analytical tangent projectors. **Real MNIST digit `1` does not supply this field.** Semantic validity or a digit label is *not* tangent ground truth. The notebook therefore correctly returns `UNAVAILABLE`, not an invented true-error bound.

Optional *independent partial check*: translation and local neighbor directions give observable geometric proxies (saved above); they **cannot certify the full tangent space**. A future project can add independently validated tangent bases and run a separate real-MNIST conformal calibration on disjoint points, but that dataset and its assumptions do not currently exist.

In [ ]:
truth_audit=dict(
  dataset='real MNIST digit 1',ground_truth_intrinsic_dimensions_available=False,
  ground_truth_full_tangent_projectors_available=False,
  true_tangent_projector_error_computable=False,
  true_error_uncertainty_calibration='UNAVAILABLE — no independently validated full tangent ground truth',
  existing_confidence_bound='Hoeffding UCB on independent bounded proxy loss among accepted anchors',
  why_synthetic_conformal_not_used='Circle-only conformal coverage failed on some unseen synthetic manifolds and cannot be assumed valid on MNIST.',
  partial_checks='Independent translation and neighbor-direction retention and local numerical stability; not true manifold error.',
  planned_resolution='Acquire independently justified tangent references for multiple real MNIST samples, define target loss, use new disjoint training/calibration/test splits and evaluate actual held-out simultaneous coverage.',
  recommendation='Treat banks as classifier-free exploratory candidates, not certified manifold tangent directions.')
(OUTPUT_DIR/'TRUE_ERROR_IDENTIFIABILITY_AUDIT.json').write_text(json.dumps(truth_audit,indent=2))
print(json.dumps(truth_audit,indent=2))

## J. Preview directions, export a self-contained evidence archive
Direction heatmaps visualize pixel-space vectors; they are **not yet generated manifold-drift images**. The archive includes frozen policy/provenance, all selected/test diagnostics, the independently measured reference scores, and direction banks for Task 3.

In [ ]:
import matplotlib.pyplot as plt, shutil
fig,axs=plt.subplots(len(new_indices),4,figsize=(12,3*len(new_indices)))
axs=np.asarray(axs).reshape(len(new_indices),4)
for i,idx in enumerate(new_indices):
    x=(ds.data[idx].float()/127.5-1).numpy()
    axs[i,0].imshow(x,cmap='gray',vmin=-1,vmax=1)
    axs[i,0].set_title(f'Source digit 1 #{idx}')
    sel=selection_df[selection_df['index']==idx].iloc[0]
    ts=[5,100]+([int(sel.selected_timestep)] if sel.status=='ACCEPT' else [None])
    for j,t in enumerate(ts,1):
        if t is None:
            axs[i,j].text(.1,.5,'ABSTAIN',fontsize=11,transform=axs[i,j].transAxes)
        else:
            packet=torch.load(BANK_DIR/f'digit1_index{idx}_t{t}_directions.pt',map_location='cpu',weights_only=True)
            bank=packet['direction_bank']
            if len(bank):
                direction=bank[0].reshape(28,28).numpy()
                scale=max(np.max(np.abs(direction)),1e-8)
                axs[i,j].imshow(direction,cmap='bwr',vmin=-scale,vmax=scale)
            else:
                axs[i,j].text(.1,.5,'No usable directions',transform=axs[i,j].transAxes)
            axs[i,j].set_title(f't={t}'+(' (selected)' if j==3 else ''))
    for a in axs[i]:a.axis('off')
plt.tight_layout()
fig.savefig(OUTPUT_DIR/'TASK2_DIRECTION_PANEL.png',dpi=160)
plt.show()

report = """# Classifier-free MNIST geometry Task 2 — honest scientific status

- **Method**: The original FIT-ranked 32-policy calibration is frozen and reused, not newly trained on evaluation images.
- **Predictor**: pretrained conditional DDPM noise estimator; score-Jacobian spectral filter rather than an exactly conservative EBM Hessian.
- **Policy calibration**: bounded translation/neighbor geometry-proxy risk. The reported UCB is not a true tangent-projector-error bound.
- **Inference**: no target classifier, surrogate classifier, or classifier gradient used.
- **Outputs**: accepted selected scales, fixed-scale baselines, matrix-free pixel-space vector banks, independent retention and local-stability diagnostics.
- **Limit**: The true MNIST tangent projectors/dimensions are unknown, so reliable true-error calibration remains an unresolved research problem.
- **Next**: Freeze image candidates from these directions; then separately assess semantic preservation and transfer on independent classifier architectures.
"""
(OUTPUT_DIR/'RESULTS_README.md').write_text(report)
ARCHIVE=shutil.make_archive('/content/MNIST_Calibrated_UCB_Task2_Evidence','zip',root_dir=str(OUTPUT_DIR))
print('RESULTS:',OUTPUT_DIR,'\nZIP:',ARCHIVE)
from IPython.display import FileLink,display
# Drive always contains the persistent files; this ZIP is a convenience.
display(FileLink(ARCHIVE))

### Interpretation for the November 3 proposal

Report **three separate things**: (1) original proxy-UCB calibration and acceptance coverage, (2) *new, disjoint* fixed-vs-selected directional measurements and any abstention, and (3) true-error uncertainty as **not established on MNIST**. Avoid calling a proxy-UCB a true-error confidence bound. If results are poor, keep the failure rather than changing thresholds after inspection. Task 3 may use fixed-scale exploratory banks even when the calibrated method abstains, as long as this status is explicit.